In [0]:
# Databricks notebook
# Purpose: Create and load the reference Delta table aux_de_para_grupo_empresa
# Source: Embedded curated mapping based on SUSEP entities and economic groups

from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, DateType, StringType, StructField, StructType, TimestampType

#  ============================================================
# 1. CONFIGURATION
#  ============================================================

database_name = "susep_gold"
table_name = "aux_de_para_grupo_empresa"
full_table_name = f"{database_name}.{table_name}"


# Regra de vigencia da proposta.
# Use 1900-01-01 para aplicar o de/para retroativamente em series historicas.
vigencia_inicio_default = "1900-01-01"
vigencia_fim_default = None
flag_ativo_default = True
versao_mapeamento_default = "v1.0"
fonte_regra_default = "Proposta interna baseada em entidades, grupos SUSEP e análise concorrencial do mercado de seguros"
write_mode = "overwrite"

print(f"Target table: {full_table_name}")


In [0]:
# ============================================================
# 2. REFERENCE DATA
# ============================================================
# Estrutura base: coenti, noenti, cogrupo, nogrupo, search_string,
# nome_empresa_consolidada_query_original, grupo_concorrencia_n1,
# grupo_concorrencia_n2, tipo_grupo, regra_aplicada,
# confianca_mapeamento, observacao, alterou_vs_query_atual.

mapping_data = [('10677',
  'ARC PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ARC PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('49760',
  'American Home Assurance Company',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AMERICAN HOME ASSURANCE COMPANY',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('21628',
  'CARDIF CAPITALIZAÇÃO S/A',
  '1203',
  'CARDIF',
  'CARDIF | CARDIF CAPITALIZAÇÃO S/A',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('6912',
  'ITAU BMG SEGURADORA S/A "EM APROVAÇÃO"',
  '35',
  'ITAÚ',
  'ITAÚ | ITAU BMG SEGURADORA S/A "EM APROVAÇÃO"',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('9938',
  'Safra Vida e Previdência S.A.',
  '345',
  'SAFRA',
  'SAFRA | SAFRA VIDA E PREVIDÊNCIA S.A.',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('3034',
  'ECC DO BRASIL CIA DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ECC DO BRASIL CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3069',
  'POTTENCIAL SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | POTTENCIAL SEGURADORA SA',
  'POTTENCIAL',
  'POTTENCIAL',
  'Pottencial',
  'Garantia / seguradora especializada',
  'Pottencial',
  'Alta',
  None,
  'NÃO'),
 ('77721',
  'Benfield do Brasil Corretora de Resseguros Ltda.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BENFIELD DO BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4561',
  'BRASILPREV NOSSO FUTURO SEGUROS E PREVIDÊNCIA S/A "em aprovação"',
  '1212',
  'MAPFRE',
  'MAPFRE | BRASILPREV NOSSO FUTURO SEGUROS E PREVIDÊNCIA S/A "EM APROVAÇÃO"',
  'MAPFRE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6602',
  'Mitsui Sumitomo Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MITSUI SUMITOMO SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3182',
  'ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A.',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A.',
  'ITAÚ UNIBANCO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'SIM'),
 ('5096',
  'PHENIX SEGURADORA S.A.',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | PHENIX SEGURADORA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('45926',
  'ARDEN REINSURANCE COMPANY LTD.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARDEN REINSURANCE COMPANY LTD.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1554',
  'EQ SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EQ SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3646',
  'Ezze Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EZZE SEGUROS S.A.',
  'OUTROS',
  'EZZE',
  'Ezze',
  'Seguradora nacional independente',
  'Ezze',
  'Alta',
  None,
  'SIM'),
 ('4367',
  'PRUDENTIAL DO BRASIL SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PRUDENTIAL DO BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2275',
  'SEGURADORA ALM S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA ALM S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21628',
  'VANGUARDACAP CAPITALIZAÇÃO S.A.',
  '825',
  'ICATU',
  'ICATU | VANGUARDACAP CAPITALIZAÇÃO S.A.',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('43419',
  'STARR INSURANCE & REINSURANCE LIMITED',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | STARR INSURANCE & REINSURANCE LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75434',
  'WILLIS CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | WILLIS CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2852',
  'AXA SEGUROS S.A.',
  '1195',
  'AXA',
  'AXA | AXA SEGUROS S.A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('4600',
  'BVIX SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BVIX SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3034',
  'ECC DO BRASIL CIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ECC DO BRASIL CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5436',
  'Junto Seguros S.A. "em aprovação" (antiga J. Malucelli Seguradora S/A)',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | JUNTO SEGUROS S.A. "EM APROVAÇÃO" (ANTIGA J. MALUCELLI SEGURAD...',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('45896',
  'GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO BRASIL LTDA',
  '27',
  'BRADESCO',
  'BRADESCO | GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO BRAS...',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('3166',
  'METLIFE VIDA E PREVIDÊNCIA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | METLIFE VIDA E PREVIDÊNCIA S/A',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('5690',
  'Companhia Excelsior de Seguros',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPANHIA EXCELSIOR DE SEGUROS',
  'OUTROS',
  'EXCELSIOR',
  'Excelsior',
  'Seguradora nacional independente',
  'Excelsior',
  'Alta',
  None,
  'SIM'),
 ('46507',
  'AXIS RE LIMITED',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AXIS RE LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('47708',
  'XL Catlin Insurance Company UK Limited',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XL CATLIN INSURANCE COMPANY UK LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3816',
  'Real Tokio Marine Vida e Previdência S.A. "em aprovação" (antiga Real Vida e ...',
  '86',
  'REAL',
  'REAL | REAL TOKIO MARINE VIDA E PREVIDÊNCIA S.A. "EM APROVAÇÃO" (ANTIGA REAL ...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('45896',
  'GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO BRASIL LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71412',
  'UNIVERSAL RE CORRETORES DE RESSEGUROS LTDA.',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | UNIVERSAL RE CORRETORES DE RESSEGUROS LTDA.',
  'MAPFRE',
  'MAPFRE / BBMAPFRE LEGADO',
  'BBMAPFRE legado não classificado',
  'Joint venture/legado',
  'Grupo SUSEP BBMAPFRE sem palavra-chave de entidade',
  'Baixa',
  'Não agrupar automaticamente tudo em BB; classificar por entidade e período.',
  'SIM'),
 ('1015',
  'SUL AMÉRICA SEGUROS DE AUTOMÓVEIS E MASSIFICADOS S.A.',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SEGUROS DE AUTOMÓVEIS E MASSIFICADOS S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('28932',
  'RIO\'S CAPITALIZAÇÃO S.A. - "em aprovação" (antiga Sul América Capitalização S...',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | RIO\'S CAPITALIZAÇÃO S.A. - "EM APROVAÇÃO" (ANTIGA SUL AMÉRICA C...',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5312',
  'COMPANHIA UNIÃO DE SEGUROS GERAIS',
  '27',
  'BRADESCO',
  'BRADESCO | COMPANHIA UNIÃO DE SEGUROS GERAIS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5363',
  'MERIDIONAL COMPANHIA DE SEGUROS GERAIS',
  '451',
  'MERIDIONAL',
  'MERIDIONAL | MERIDIONAL COMPANHIA DE SEGUROS GERAIS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MERIDIONAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5436',
  'J. Malucelli Seguradora S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | J. MALUCELLI SEGURADORA S/A',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('5762',
  'Santos Cia.Segs de Gar.e Crédito S/A(San',
  '1112',
  'SANTOS',
  'SANTOS | SANTOS CIA.SEGS DE GAR.E CRÉDITO S/A(SAN',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5819',
  'AMERICAN LIFE COMPANHIA DE SEGUROS',
  '1031',
  'BLUE LIFE',
  'BLUE LIFE | AMERICAN LIFE COMPANHIA DE SEGUROS',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('5932',
  'ATLANTICA - BRADESCO SEGUROS S.A',
  '27',
  'BRADESCO',
  'BRADESCO | ATLANTICA - BRADESCO SEGUROS S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5975',
  'BCN SEGURADORA S.A.',
  '507',
  'BCN',
  'BCN | BCN SEGURADORA S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5983',
  'UNIBANCO SEGUROS S/A',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO SEGUROS S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5991',
  'SEGURADORA BRASILEIRA RURAL S/A',
  '1229',
  'UBF',
  'UBF | SEGURADORA BRASILEIRA RURAL S/A',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6190',
  'TOKIO MARINE SEGURADORA S.A. "EM APROVAÇÃO" (ANTIGA REAL SEGUROS S.A.)',
  '1222',
  'TOKIO MARINE',
  'TOKIO MARINE | TOKIO MARINE SEGURADORA S.A. "EM APROVAÇÃO" (ANTIGA REAL SEGUR...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'CANADA LIFE PREVIDENCIA E SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CANADA LIFE PREVIDENCIA E SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6297',
  'AIG LIFE COMPANHIA DE SEGUROS',
  '108',
  'INTERAMERICANA',
  'INTERAMERICANA | AIG LIFE COMPANHIA DE SEGUROS',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('6351',
  'METROPOLITAN LIFE SEGUROS E PREVID-NCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | METROPOLITAN LIFE SEGUROS E PREVID-NCIA',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('6360',
  'Kyoei do Brasil Companhia de Seguros',
  '523',
  'KYOEI',
  'KYOEI | KYOEI DO BRASIL COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'KYOEI',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6416',
  'YASUDA SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | YASUDA SEGUROS S/A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6459',
  'Cia de Seguros Inter-Atlantico',
  '591',
  'INTER-ATLANTICO',
  'INTER-ATLANTICO | CIA DE SEGUROS INTER-ATLANTICO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INTER-ATLANTICO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6483',
  'Winterthur International Brasil Segurado',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | WINTERTHUR INTERNATIONAL BRASIL SEGURADO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6513',
  'ACE SEGURADORA S.A.',
  '1189',
  'ACE',
  'ACE | ACE SEGURADORA S.A.',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('6530',
  'VIRGINIA SURETY COMPANHIA DE SEGUROS DO BRASIL S.A. "em aprovação" (antiga CO...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIRGINIA SURETY COMPANHIA DE SEGUROS DO BRASIL S.A. "EM APROVA...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6734',
  'AIG BRASIL INTERAMERICANA CIA DE SEGS GE',
  '108',
  'INTERAMERICANA',
  'INTERAMERICANA | AIG BRASIL INTERAMERICANA CIA DE SEGS GE',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('6777',
  'APS SEGURADORA S.A. (em aprovação) CAOA',
  '671',
  'REUNIDAS',
  'REUNIDAS | APS SEGURADORA S.A. (EM APROVAÇÃO) CAOA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'REUNIDAS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6777',
  'APS SEGURADORA S.A. (em aprovação) CAOA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | APS SEGURADORA S.A. (EM APROVAÇÃO) CAOA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6840',
  'Unibanco AIG Previdência SA',
  '701',
  'PREVER',
  'PREVER | UNIBANCO AIG PREVIDÊNCIA SA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6939',
  'Seguradora SEASUL S.A.',
  '1215',
  'METROPOLITAN LIFE',
  'METROPOLITAN LIFE | SEGURADORA SEASUL S.A.',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('8141',
  'CAIXA VIDA E PREVIDÊNCIA S/A" EM APROV "',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXA VIDA E PREVIDÊNCIA S/A" EM APROV "',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('10081',
  'NEWPREV PREVIDÊNCIA PRIVADA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NEWPREV PREVIDÊNCIA PRIVADA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10103',
  'ARCESP PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARCESP PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10120',
  'EQUATORIAL PREVIDENCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EQUATORIAL PREVIDENCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10669',
  'RSPP PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | RSPP PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11045',
  'BRASILPREV SEGUROS E PREVIDÊNCIA S/A',
  '78',
  'BRASIL',
  'BRASIL | BRASILPREV SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('20273',
  'Santos Capitalização S/A',
  '1112',
  'SANTOS',
  'SANTOS | SANTOS CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70203',
  'BSR BRASIL SPECIAL RISKS CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BSR BRASIL SPECIAL RISKS CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78778',
  'ORYPABA RIO ADM E CORRETAGEM DE SEGUROS E RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ORYPABA RIO ADM E CORRETAGEM DE SEGUROS E RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5002',
  'FEDERAL DE SEGUROS S/A',
  '132',
  'FEDERAL',
  'FEDERAL | FEDERAL DE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'FEDERAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5061',
  'SEGURADORA OCEÂNICA S/A',
  '451',
  'MERIDIONAL',
  'MERIDIONAL | SEGURADORA OCEÂNICA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MERIDIONAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5657',
  'VANGUARDA COMPANHIA DE SEGUROS GERAIS',
  '825',
  'ICATU',
  'ICATU | VANGUARDA COMPANHIA DE SEGUROS GERAIS',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('11304',
  'Luterprev - Entidade Luterana de Previde',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | LUTERPREV - ENTIDADE LUTERANA DE PREVIDE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21318',
  'INVEST CAPITALIZAÇÃO S/A',
  '710',
  'RURAL',
  'RURAL | INVEST CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('34819',
  'Austral Resseguradora S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AUSTRAL RESSEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71625',
  'BOWRING MARSH CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BOWRING MARSH CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6866',
  'BRADESCO VIDA E PREVIDÊNCIA S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO VIDA E PREVIDÊNCIA S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('41491',
  'HANNOVER RÜCK SE',
  '1210',
  'HANNOVER',
  'HANNOVER | HANNOVER RÜCK SE',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('1147',
  'FEDERAL VIDA E PREVIDÊNCIA S/A',
  '132',
  'FEDERAL',
  'FEDERAL | FEDERAL VIDA E PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'FEDERAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1970',
  'UNIMED SEGUROS PATRIMONIAIS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UNIMED SEGUROS PATRIMONIAIS S.A.',
  'UNIMED',
  'UNIMED',
  'Unimed',
  'Saúde/operadora/cooperativa',
  'Unimed',
  'Alta',
  None,
  'NÃO'),
 ('38253',
  'ACE RESSEGURADORA S/A',
  '1189',
  'ACE',
  'ACE | ACE RESSEGURADORA S/A',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('4812',
  'Euler do Brasil Seguros de Crédito à Exp',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER DO BRASIL SEGUROS DE CRÉDITO À EXP',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6491',
  'Alvorada Vida S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | ALVORADA VIDA S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('14818',
  'UNIMED PREVIDENCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UNIMED PREVIDENCIA PRIVADA',
  'UNIMED',
  'UNIMED',
  'Unimed',
  'Saúde/operadora/cooperativa',
  'Unimed',
  'Alta',
  None,
  'NÃO'),
 ('29319',
  'Santander Capitalização S/A',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER CAPITALIZAÇÃO S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('1970',
  'UNIMED SEGUROS PATRIMONIAIS S.A.',
  '795',
  'UNIMED',
  'UNIMED | UNIMED SEGUROS PATRIMONIAIS S.A.',
  'UNIMED',
  'UNIMED',
  'Unimed',
  'Saúde/operadora/cooperativa',
  'Unimed',
  'Alta',
  None,
  'NÃO'),
 ('75761',
  'BESSO RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BESSO RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3093',
  'CONSÓRCIO DOS SEGUROS DPVAT',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CONSÓRCIO DOS SEGUROS DPVAT',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('36099',
  'AXA XL RESSEGUROS S.A.',
  '1195',
  'AXA',
  'AXA | AXA XL RESSEGUROS S.A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('2747',
  'GALAPAGOS CAPITAL SOCIEDADE SEGURADORA DE PROPOSITO ESPECIFICO S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GALAPAGOS CAPITAL SOCIEDADE SEGURADORA DE PROPOSITO ESPECIFIC...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('31551',
  'SCOR Brasil Resseguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SCOR BRASIL RESSEGUROS S.A.',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('2607',
  'FELSEN SEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FELSEN SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4227',
  'STG SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | STG SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4227',
  'STG SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | STG SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5720',
  'Yasuda Maritima Seguros S/A (em aprovação)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | YASUDA MARITIMA SEGUROS S/A (EM APROVAÇÃO)',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('2879',
  'Comprev Seguros e Previdência S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMPREV SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('79006',
  'LOCKTON RE BRASIL CORRETORA DE RESSEGUROS EIRELI',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LOCKTON RE BRASIL CORRETORA DE RESSEGUROS EIRELI',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6696',
  'AXA CORPORATE SOLUTIONS SEGUROS S.A. "em aprovação" (antiga SUL AMÉRICA COMPA...',
  '1195',
  'AXA',
  'AXA | AXA CORPORATE SOLUTIONS SEGUROS S.A. "EM APROVAÇÃO" (ANTIGA SUL AMÉRICA...',
  'OUTROS',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'SIM'),
 ('21318',
  'INVEST CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | INVEST CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23540',
  'APLUB Capitalização S. A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | APLUB CAPITALIZAÇÃO S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2119',
  'BVA SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BVA SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2763',
  'SEGURADORA DE CRÉDITO DO BRASIL S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEGURADORA DE CRÉDITO DO BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6858',
  'MAPFRE AFFINITY SEGURADORA S.A. "em aprovação"',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE AFFINITY SEGURADORA S.A. "EM APROVAÇÃO"',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5401',
  'BBM Cia. de Seguros',
  '272',
  'SEGUROS DA BAHIA',
  'SEGUROS DA BAHIA | BBM CIA. DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SEGUROS DA BAHIA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5487',
  'SANTANDER BRASIL SEGUROS S/A',
  '370',
  'NOROESTE',
  'NOROESTE | SANTANDER BRASIL SEGUROS S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5533',
  'FINASA SEGURADORA S/A',
  '27',
  'BRADESCO',
  'BRADESCO | FINASA SEGURADORA S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5614',
  'Sul América Santa Cruz Seguros S/A',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SANTA CRUZ SEGUROS S/A',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5631',
  'SASSE CAIXA SEGUROS',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | SASSE CAIXA SEGUROS',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('5789',
  'amil seguradora s/a',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AMIL SEGURADORA S/A',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('5908',
  'GENERALI DO BRASIL CIA NAC DE SEGUROS',
  '213',
  'GENERALLI',
  'GENERALLI | GENERALI DO BRASIL CIA NAC DE SEGUROS',
  'GENERALI',
  'GENERALI',
  'Generali',
  'Seguradora global multilinha',
  'Generali',
  'Alta',
  None,
  'NÃO'),
 ('5991',
  'Swiss Re Corporate Solutions Brasil Seguros S/A "em aprovação"  (antiga UBF S...',
  '1229',
  'UBF',
  'UBF | SWISS RE CORPORATE SOLUTIONS BRASIL SEGUROS S/A "EM APROVAÇÃO"  (ANTIGA...',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6050',
  'MAXMED SEGURADORA S/A  ( MAXLIFE EM APRO',
  '1023',
  'NOBRE',
  'NOBRE | MAXMED SEGURADORA S/A  ( MAXLIFE EM APRO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NOBRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6131',
  'COMPANHIA MUTUAL DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMPANHIA MUTUAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6181',
  'Brasil Veiculos Companhia de Seguros',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | BRASIL VEICULOS COMPANHIA DE SEGUROS',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6190',
  'REAL SEGUROS S.A.',
  '86',
  'REAL',
  'REAL | REAL SEGUROS S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6220',
  'SUL AMÉRICA SEGUROS DE VIDA E PREVIDÊNCI',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SEGUROS DE VIDA E PREVIDÊNCI',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'PACTUAL PREVID-NCIA E SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PACTUAL PREVID-NCIA E SEGUROS S/A',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVIDÊNCIA PRIVADA S.A.',
  '78',
  'BRASIL',
  'BRASIL | AGF VIDA E PREVIDÊNCIA PRIVADA S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVIDÊNCIA S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | AGF VIDA E PREVIDÊNCIA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6521',
  'CAIXAGERAL S/A SEGURADORA',
  '949',
  'CAIXAGERAL',
  'CAIXAGERAL | CAIXAGERAL S/A SEGURADORA',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('6564',
  'Santander Brasil Seguros S/A ( em Aprov) Antiga Santander Banespa Seguros S/A',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER BRASIL SEGUROS S/A ( EM APROV) ANTIGA SANTANDER BANESPA...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6777',
  'CAOA SEGUROS DO BRASIL S/A',
  '671',
  'REUNIDAS',
  'REUNIDAS | CAOA SEGUROS DO BRASIL S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'REUNIDAS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6777',
  'APS SEGURADORA S.A.',
  '1202',
  'CAOA',
  'CAOA | APS SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CAOA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6921',
  'Rural Seguradora SA',
  '710',
  'RURAL',
  'RURAL | RURAL SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6998',
  'AUREA SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUREA SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10472',
  'GEPLAN SOCIEDADE DE PREVIDENCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GEPLAN SOCIEDADE DE PREVIDENCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDENCIA " em aprovação" (antiga SOCIEDADE AUXILIADORA )',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDENCIA " EM APROVAÇÃO" (ANTIGA SOCIEDADE AUXI...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDENCIA " em aprovação " (antiga SOCIEDADE AUXILIADORA)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDENCIA " EM APROVAÇÃO " (ANTIGA SOCIEDADE AUX...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('19291',
  'Nossa Caixa Previdência S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NOSSA CAIXA PREVIDÊNCIA S.A.',
  'CAIXA SEGURIDADE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Nossa Caixa legado',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Média',
  'Banco Nossa Caixa é legado histórico; não tratar como Caixa Econômica. Validar período societário.',
  'SIM'),
 ('22764',
  'CREFICAP CAPITALIZAÇÃO S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CREFICAP CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('25712',
  'REAL CAPITALIZAÇÃO S/A',
  '86',
  'REAL',
  'REAL | REAL CAPITALIZAÇÃO S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('32875',
  'ZURICH RESSEGURADORA BRASIL S/A',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | ZURICH RESSEGURADORA BRASIL S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('49760',
  'American Home Assurance Company',
  '1191',
  'UNIBANCO/AIG',
  'UNIBANCO/AIG | AMERICAN HOME ASSURANCE COMPANY',
  'ITAÚ UNIBANCO',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('45896',
  'GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO BRASIL',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4359',
  'Euler Hermes Seguros de Crédito S.A.',
  '1228',
  'EULER HERMES',
  'EULER HERMES | EULER HERMES SEGUROS DE CRÉDITO S.A.',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('2101',
  'MONGERAL AEGON SEGUROS E PREVIDÊNCIA S. A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MONGERAL AEGON SEGUROS E PREVIDÊNCIA S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3271',
  'SEGURADORA LÍDER DO CONSÓRCIO DO SEGURO DPVAT S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA LÍDER DO CONSÓRCIO DO SEGURO DPVAT S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6378',
  'VIDA SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VIDA SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10081',
  'NEWPREV PREVIDENCIA PRIVADA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NEWPREV PREVIDENCIA PRIVADA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('30201',
  'TERRA BRASIS RESSEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TERRA BRASIS RESSEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('26026',
  'LIDERANÇA CAPITALIZAÇÃO S/A',
  '1211',
  'GRUPO SILVIO SANTOS',
  'GRUPO SILVIO SANTOS | LIDERANÇA CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'GRUPO SILVIO SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6751',
  'ROYAL & SUNALLIANCE SEGUROS (BRASIL) S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ROYAL & SUNALLIANCE SEGUROS (BRASIL) S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6467',
  'Alfa Seguradora S.A.',
  '1192',
  'ALFA',
  'ALFA | ALFA SEGURADORA S.A.',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('1431',
  'XL Seguros Brasil S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XL SEGUROS BRASIL S.A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('2119',
  'BVA SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BVA SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77721',
  'BENFIELD DO BRASIL CORRETORA DE RESSEGUROS LTDA.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BENFIELD DO BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5096',
  'UNIBANCO AIG VIDA E PREVIDÊNCIA S/A',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO AIG VIDA E PREVIDÊNCIA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('2101',
  'MONGERAL S/A SEGUROS E PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MONGERAL S/A SEGUROS E PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23345',
  'ATLANTICA CAPITALIZAÇÃO S/A',
  '27',
  'BRADESCO',
  'BRADESCO | ATLANTICA CAPITALIZAÇÃO S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('38873',
  'J.MALUCELLI RESSEGURADORA S/A',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | J.MALUCELLI RESSEGURADORA S/A',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('39764',
  'MARKEL RESSEGURADORA DO BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MARKEL RESSEGURADORA DO BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6998',
  'CESCEBRASIL SEGUROS DE GARANTIAS E CRÉDITO S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CESCEBRASIL SEGUROS DE GARANTIAS E CRÉDITO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('47082',
  'HDI-GERLING WELT SERVICE AG',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HDI-GERLING WELT SERVICE AG',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5096',
  'ITAÚ VIDA E PREVIDÊNCIA S/A',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ VIDA E PREVIDÊNCIA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5436',
  'J. Malucelli Seguradora S/A',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | J. MALUCELLI SEGURADORA S/A',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('6122',
  'FATOR SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FATOR SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('45420',
  'Catlin Re Switzerland Ltd',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CATLIN RE SWITZERLAND LTD',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10707',
  'MBM Previdência Complementar',
  '1121',
  'MBM',
  'MBM | MBM PREVIDÊNCIA COMPLEMENTAR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MBM',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10987',
  'SOCIEDADE CAXIENSE DE MÚTUO SOCORRO - PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOCIEDADE CAXIENSE DE MÚTUO SOCORRO - PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77747',
  'GALLAGHER RE LATIN AMERICA CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GALLAGHER RE LATIN AMERICA CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5118',
  'TRADITIO COMPANHIA DE SEGUROS',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | TRADITIO COMPANHIA DE SEGUROS',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('36099',
  'AXA CORPORATE SOLUTIONS BRASIL E AMÉRICA LATINA RESSEGUROS S.A.',
  '1195',
  'AXA',
  'AXA | AXA CORPORATE SOLUTIONS BRASIL E AMÉRICA LATINA RESSEGUROS S.A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('4618',
  'ITAÚ SEGUROS CORPORATIVOS S/A',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ SEGUROS CORPORATIVOS S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('3298',
  'Mapfre Seguradora de Garantias e Crédito S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE SEGURADORA DE GARANTIAS E CRÉDITO S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('3221',
  'ANDRINA SOCIEDADE SEGURADORA DE PROPOSITO ESPECIFICO S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ANDRINA SOCIEDADE SEGURADORA DE PROPOSITO ESPECIFICO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1864',
  'TAAMIN SEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TAAMIN SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('45004',
  'ROYAL & SUN ALLIANCE INSURANCE LTD',
  '515',
  'SUN ALLIANCE',
  'SUN ALLIANCE | ROYAL & SUN ALLIANCE INSURANCE LTD',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SUN ALLIANCE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('22926',
  'BB Capitalização S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BB CAPITALIZAÇÃO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3166',
  'METLIFE VIDA E PREVIDÊNCIA S/A "em aprovação" (antiga CITIINSURANCE DO BRASIL...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | METLIFE VIDA E PREVIDÊNCIA S/A "EM APROVAÇÃO" (ANTIGA CITIINSU...',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('6238',
  'MAPFRE SEGUROS GERAIS S.A. "em aprovação"',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE SEGUROS GERAIS S.A. "EM APROVAÇÃO"',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('1775',
  'ACE SEGUROS SOLUÇÕES CORPORATIVAS S/A (em aprovação)',
  '1189',
  'ACE',
  'ACE | ACE SEGUROS SOLUÇÕES CORPORATIVAS S/A (EM APROVAÇÃO)',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('6912',
  'ITAU BMG SEGURADORA SA "EM APROVAÇÃO"',
  '35',
  'ITAÚ',
  'ITAÚ | ITAU BMG SEGURADORA SA "EM APROVAÇÃO"',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('76767',
  'Som.us do Brasil Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SOM.US DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('24872',
  'XS4 CAPITALIZAÇÃO S.A.',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | XS4 CAPITALIZAÇÃO S.A.',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('5096',
  'ITAÚ VIDA E PREVIDÊNCIA S/A',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | ITAÚ VIDA E PREVIDÊNCIA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5231',
  'BANCRED SEGURADORA',
  '1147',
  'CRUZEIRO DO SUL',
  'CRUZEIRO DO SUL | BANCRED SEGURADORA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5495',
  'Cia. Seguros Minas-Brasil',
  '531',
  'ZURICH',
  'ZURICH | CIA. SEGUROS MINAS-BRASIL',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('5541',
  'HSBC SEGURO SAÚDE S.A.',
  '1204',
  'HSBC',
  'HSBC | HSBC SEGURO SAÚDE S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5622',
  'Brasilsaude Companhia de Seguros',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | BRASILSAUDE COMPANHIA DE SEGUROS',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5665',
  'MAPFRE VERA CRUZ VIDA S.A. - (EM APROVAÇÃO)',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ VIDA S.A. - (EM APROVAÇÃO)',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5762',
  'Santos Companhia de Seguros',
  '1112',
  'SANTOS',
  'SANTOS | SANTOS COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5843',
  'Indiana Seguros S/A',
  '434',
  'INDIANA',
  'INDIANA | INDIANA SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5878',
  'SULINA SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SULINA SEGURADORA S/A',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5941',
  'QBE BRASIL SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | QBE BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5967',
  'AVS SEGURADORA S/A (EM APROVAÇÃO - AVG S',
  '1074',
  'SAMCIL',
  'SAMCIL | AVS SEGURADORA S/A (EM APROVAÇÃO - AVG S',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SAMCIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5991',
  'SEG.BRAS.RURAL S/A(EM APROVAÇÃO) ANTIGA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEG.BRAS.RURAL S/A(EM APROVAÇÃO) ANTIGA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5991',
  'UBF SEGUROS S/A.',
  '1229',
  'UBF',
  'UBF | UBF SEGUROS S/A.',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6041',
  'PARANA CIA DE SEGUROS',
  '35',
  'ITAÚ',
  'ITAÚ | PARANA CIA DE SEGUROS',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6050',
  'MAXLIFE SEG DO BR S/A (EM APROV) ANTIGA',
  '1104',
  'MAXMED',
  'MAXMED | MAXLIFE SEG DO BR S/A (EM APROV) ANTIGA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MAXMED',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6190',
  'REAL SEGUROS S.A. (REAL PREVID E SEGUROS S.A) em aprovação',
  '86',
  'REAL',
  'REAL | REAL SEGUROS S.A. (REAL PREVID E SEGUROS S.A) EM APROVAÇÃO',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'IH CIA DE SEGUROS E PREVIDÊNCIA',
  '825',
  'ICATU',
  'ICATU | IH CIA DE SEGUROS E PREVIDÊNCIA',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'COMPANHIA BRASILEIRA DE SEGUROS E PREVIDÊNCIA "em aprovação" (antiga IH CIA. ...',
  '825',
  'ICATU',
  'ICATU | COMPANHIA BRASILEIRA DE SEGUROS E PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA ...',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'IH CIA DE SEGUROS E PREVIDÊNCIA "em aprovação" (Antiga denominção da Canadá L...',
  '825',
  'ICATU',
  'ICATU | IH CIA DE SEGUROS E PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA DENOMINÇÃO DA ...',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'PACTUAL PREVIDÊNCIA E SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PACTUAL PREVIDÊNCIA E SEGUROS S/A',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('6343',
  'Sol de Seguros S/A',
  '132',
  'FEDERAL',
  'FEDERAL | SOL DE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'FEDERAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6351',
  'METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('6483',
  'XL Insurance Brazl Seg S/A"em aprovação"',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | XL INSURANCE BRAZL SEG S/A"EM APROVAÇÃO"',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6491',
  'BBV PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  '1199',
  'BBV',
  'BBV | BBV PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6530',
  'COMBINED SEGUROS BRASIL S/A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMBINED SEGUROS BRASIL S/A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6572',
  'HANNOVER INTERNATIONAL SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HANNOVER INTERNATIONAL SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6572',
  'HDI SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HDI SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6572',
  'HDI SEGUROS S.A.',
  '1210',
  'HANNOVER',
  'HANNOVER | HDI SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6611',
  'BEMGE SEGURADORA S/A',
  '647',
  'BEMGE',
  'BEMGE | BEMGE SEGURADORA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6688',
  'COSESP - CIA DE SEG DO EST. DE S.P.',
  '159',
  'COSESP',
  'COSESP | COSESP - CIA DE SEG DO EST. DE S.P.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'COSESP',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6734',
  'AIG Brasil Companhia de Seguros',
  '108',
  'INTERAMERICANA',
  'INTERAMERICANA | AIG BRASIL COMPANHIA DE SEGUROS',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('6831',
  'PESSOAL CIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PESSOAL CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6882',
  'COMPANHIA GERAL DE SEGUROS',
  '710',
  'RURAL',
  'RURAL | COMPANHIA GERAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6904',
  'SOMA SEGURADORA S/A',
  '736',
  'SOMA',
  'SOMA | SOMA SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SOMA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('8737',
  'AIG BRASIL COMPANHIA DE SEGUROS',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | AIG BRASIL COMPANHIA DE SEGUROS',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('10359',
  'CORRFA PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CORRFA PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10634',
  'Mongeral Previdência Privada',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MONGERAL PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10961',
  'PREVICORP Previdência Privada',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PREVICORP PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23345',
  'ATLANTICA CAPITALIZAÇÃO S/A/BCN CAPITALI',
  '27',
  'BRADESCO',
  'BRADESCO | ATLANTICA CAPITALIZAÇÃO S/A/BCN CAPITALI',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('24813',
  'SINOSSERRA CAPITALIZAÇÃO S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SINOSSERRA CAPITALIZAÇÃO S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('28932',
  'SATPAR CAPITALIZAÇÃO S.A.',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SATPAR CAPITALIZAÇÃO S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('48275',
  'ZURICH INSURANCE COMPANY',
  '531',
  'ZURICH',
  'ZURICH | ZURICH INSURANCE COMPANY',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('26158',
  'KIRTON CAPITALIZAÇÃO S.A. (Atual denominação da HSBC Empresa de Capitalização...',
  '27',
  'BRADESCO',
  'BRADESCO | KIRTON CAPITALIZAÇÃO S.A. (ATUAL DENOMINAÇÃO DA HSBC EMPRESA DE CA...',
  'BRADESCO SEGUROS',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'SIM'),
 ('3671',
  'USEBENS SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | USEBENS SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4812',
  'Euler Hermes Seguros de Credito à Exportação S.A.',
  '1228',
  'EULER HERMES',
  'EULER HERMES | EULER HERMES SEGUROS DE CREDITO À EXPORTAÇÃO S.A.',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('5070',
  'Santander Seguros S/A',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER SEGUROS S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5886',
  'PORTO SEGURO CIA DE SEGUROS GERAIS',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO CIA DE SEGUROS GERAIS',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('11126',
  'PECÚLIO UNIÃO PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PECÚLIO UNIÃO PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21334',
  'ICATU CAPITALIZAÇÃO S.A.',
  '825',
  'ICATU',
  'ICATU | ICATU CAPITALIZAÇÃO S.A.',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('74811',
  'UIB RE BRASIL CORRETORA DE RESSSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UIB RE BRASIL CORRETORA DE RESSSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2798',
  'ARGO SEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARGO SEGUROS BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6688',
  'COSESP - CIA DE SEG DO EST. DE S.P.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COSESP - CIA DE SEG DO EST. DE S.P.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5142',
  'ICATU SEGUROS S.A',
  '825',
  'ICATU',
  'ICATU | ICATU SEGUROS S.A',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('2020',
  'CRÉDITO y CAUCIÓN SEGURADORA DE CRÉDITO E GARANTIAS S.A.',
  '1231',
  'CRÉDITO Y CAUCIÓN',
  'CRÉDITO Y CAUCIÓN | CRÉDITO Y CAUCIÓN SEGURADORA DE CRÉDITO E GARANTIAS S.A.',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('43290',
  'XL Re latin America Ltd',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XL RE LATIN AMERICA LTD',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('47724',
  'MITSUI SUMITOMO INSURANCE COMPANY, LIMITED',
  '1216',
  'MITSUI MARINE',
  'MITSUI MARINE | MITSUI SUMITOMO INSURANCE COMPANY, LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MITSUI MARINE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2721',
  'CRÉDITO Y CAUCIÓN SEGURADORA DE CRÉDITO À EXPORTAÇÃO S.A',
  '1231',
  'CRÉDITO Y CAUCIÓN',
  'CRÉDITO Y CAUCIÓN | CRÉDITO Y CAUCIÓN SEGURADORA DE CRÉDITO À EXPORTAÇÃO S.A',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('5231',
  'BCS SEGUROS S/A',
  '1147',
  'CRUZEIRO DO SUL',
  'CRUZEIRO DO SUL | BCS SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CRUZEIRO DO SUL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5070',
  'BOZANO, SIMONSEN SEGURADORA S/A',
  '884',
  'BOZANO SIMONSEN',
  'BOZANO SIMONSEN | BOZANO, SIMONSEN SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BOZANO SIMONSEN',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('27995',
  'HSBC CAPITALIZAÇÃO (BRASIL) S.A.',
  '1204',
  'HSBC',
  'HSBC | HSBC CAPITALIZAÇÃO (BRASIL) S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('28932',
  'SUL AMÉRICA CAPITALIZAÇÃO S.A. - SULACAP',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA CAPITALIZAÇÃO S.A. - SULACAP',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('44661',
  'SCOR GLOBAL LIFE U.S. RE INSURANCE COMPANY',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SCOR GLOBAL LIFE U.S. RE INSURANCE COMPANY',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('6173',
  'INVESTPREV SEGUROS E PREVIDÊNCIA S/A',
  '710',
  'RURAL',
  'RURAL | INVESTPREV SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('5185',
  'LIBERTY SEGUROS S/A',
  '1232',
  'LIBERTY',
  'LIBERTY | LIBERTY SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5843',
  'Indiana Seguros S/A',
  '1232',
  'LIBERTY',
  'LIBERTY | INDIANA SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('70939',
  'NAUSCH HOGAN & MURRAY BRASIL CORRETORA DE RESSEGUROS LTDA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NAUSCH HOGAN & MURRAY BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1058',
  'BRADESCO AUTO/RE COMPANHIA DE SEGUROS',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO AUTO/RE COMPANHIA DE SEGUROS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6751',
  'SEGUROS SURA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEGUROS SURA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6912',
  'ITAUSEG SEGURADORA S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAUSEG SEGURADORA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('2798',
  'AKAD SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AKAD SEGUROS S.A.',
  'OUTROS',
  'AKAD',
  'Akad',
  'Seguradora nacional independente',
  'Akad',
  'Alta',
  None,
  'SIM'),
 ('6785',
  'Brasilseg Companhia de Seguros',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | BRASILSEG COMPANHIA DE SEGUROS',
  'MAPFRE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('38873',
  'JUNTO RESSEGUROS S.A.',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | JUNTO RESSEGUROS S.A.',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('37893',
  'FM RESSEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FM RESSEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71625',
  'CARPENTER MARSH FAC BRASIL CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CARPENTER MARSH FAC BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('48186',
  'MÜNCHENER RÜCKVERSICHERUNGS-GESSELLSCHAFT AKTIENGESELLSCHAFT IN MÜNCHEN',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MÜNCHENER RÜCKVERSICHERUNGS-GESSELLSCHAFT AKTIENGESELLSCHAFT ...',
  'OUTROS',
  'MUNICH RE',
  'Munich Re',
  'Resseguro',
  'Munich Re',
  'Alta',
  None,
  'SIM'),
 ('6181',
  'Brasilveículos Companhia de Seguros',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | BRASILVEÍCULOS COMPANHIA DE SEGUROS',
  'MAPFRE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('3166',
  'CITIINSURANCE DO BRASIL VIDA E PREV S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CITIINSURANCE DO BRASIL VIDA E PREV S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4618',
  'ITAÚ XL SEGUROS CORPORATIVOS S.A. (Nova Razão Social em Aprovação)',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ XL SEGUROS CORPORATIVOS S.A. (NOVA RAZÃO SOCIAL EM APROVAÇÃO)',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('3603',
  'SIMPLE2U SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SIMPLE2U SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3964',
  'LIBERTY MUTUAL SURETY BRASIL SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LIBERTY MUTUAL SURETY BRASIL SEGUROS S.A.',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('15580',
  'BMC PREVIDÊNCIA PRIVADA S/A',
  '27',
  'BRADESCO',
  'BRADESCO | BMC PREVIDÊNCIA PRIVADA S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5908',
  'GENERALI  BRASIL  SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | GENERALI  BRASIL  SEGUROS S.A.',
  'GENERALI',
  'GENERALI',
  'Generali',
  'Seguradora global multilinha',
  'Generali',
  'Alta',
  None,
  'NÃO'),
 ('5185',
  'YLM SEGUROS S.A.',
  '1230',
  'TALANX AG',
  'TALANX AG | YLM SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('4359',
  'Euler Hermes Seguros de Crédito S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER HERMES SEGUROS DE CRÉDITO S.A.',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('75761',
  'Besso Re Brasil Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BESSO RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2020',
  'CRÉDITO y CAUCIÓN SEGURADORA DE CRÉDITO E GARANTIAS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CRÉDITO Y CAUCIÓN SEGURADORA DE CRÉDITO E GARANTIAS S.A.',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('47082',
  'HDI GLOBAL NETWORK AG (em aprovação)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HDI GLOBAL NETWORK AG (EM APROVAÇÃO)',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5142',
  'ICATU SEGUROS S.A "em aprovação" (antifga Icatu Hartford Seguros S/A)',
  '825',
  'ICATU',
  'ICATU | ICATU SEGUROS S.A "EM APROVAÇÃO" (ANTIFGA ICATU HARTFORD SEGUROS S/A)',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('5266',
  'Motor Union Seguros S/A',
  '477',
  'MOTOR UNION',
  'MOTOR UNION | MOTOR UNION SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MOTOR UNION',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5312',
  'COMPANHIA UNI¦O DE SEGUROS GERAIS',
  '27',
  'BRADESCO',
  'BRADESCO | COMPANHIA UNI¦O DE SEGUROS GERAIS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5347',
  'EDEL SEGURADORA S.A.',
  '906',
  'EDEL',
  'EDEL | EDEL SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'EDEL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5355',
  'AXA SEGUROS BRASIL S/A.',
  '1195',
  'AXA',
  'AXA | AXA SEGUROS BRASIL S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('5916',
  'REAL SEGURADORA S.A',
  '86',
  'REAL',
  'REAL | REAL SEGURADORA S.A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5967',
  'AVS SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AVS SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6009',
  'BANERJ SEGUROS S/A',
  '337',
  'BANERJ',
  'BANERJ | BANERJ SEGUROS S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6017',
  'CENTAURO VIDA E PREVIDÊNCIA S. A. "EM APROVAÇÃO" (ANTIGA CENTAURO SEGURADORA ...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CENTAURO VIDA E PREVIDÊNCIA S. A. "EM APROVAÇÃO" (ANTIGA CENTA...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6041',
  'PARANA CIA DE SEGUROS',
  '43',
  'BAMERINDUS',
  'BAMERINDUS | PARANA CIA DE SEGUROS',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6131',
  'COMPANHIA MUTUAL DE SEGUROS',
  '1209',
  'MUTUAL',
  'MUTUAL | COMPANHIA MUTUAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MUTUAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6157',
  'MINAS BRASIL VEICULOS SEGURADORA S/A',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | MINAS BRASIL VEICULOS SEGURADORA S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6181',
  'Brasilveículos Companhia de Seguros',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | BRASILVEÍCULOS COMPANHIA DE SEGUROS',
  'SULAMÉRICA',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6238',
  'MAPFRE VERA CRUZ SEGURADORA S/A',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | MAPFRE VERA CRUZ SEGURADORA S/A',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'CANADA LIFE PREVIDENCIA E SEGUROS S/A',
  '825',
  'ICATU',
  'ICATU | CANADA LIFE PREVIDENCIA E SEGUROS S/A',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6378',
  'VIDA SEGURADORA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIDA SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6416',
  'YASUDA SEGUROS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | YASUDA SEGUROS S.A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6491',
  'BBV PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BBV PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6513',
  'ACE SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ACE SEGURADORA S.A.',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('6491',
  'BBVA PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BBVA PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6530',
  'VIRGINIA SURETY COMPANHIA DE SEGUROS DO BRASIL  "em aprovação" (antiga COMBIN...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIRGINIA SURETY COMPANHIA DE SEGUROS DO BRASIL  "EM APROVAÇÃO"...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6696',
  'GERLING SUL AMÉRICA S.A. SEGUROS INDUSTR',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | GERLING SUL AMÉRICA S.A. SEGUROS INDUSTR',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6807',
  'ADRESS SEGUROS E PREVIDENCIA S/A',
  '639',
  'FIDÚCIA',
  'FIDÚCIA | ADRESS SEGUROS E PREVIDENCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'FIDÚCIA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6882',
  'RURAL CAPITALIZAÇÃO S/A. "EM APROVAÇÃO"',
  '710',
  'RURAL',
  'RURAL | RURAL CAPITALIZAÇÃO S/A. "EM APROVAÇÃO"',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('8737',
  'AIU SEGUROS SA "em aprovação" (antiga AIG BRASIL COMPANHIA DE SEGUROS)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AIU SEGUROS SA "EM APROVAÇÃO" (ANTIGA AIG BRASIL COMPANHIA DE...',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('8737',
  'CHARTIS SEGUROS BRASIL S.A.  "em aprovação" (antiga AIU SEGUROS S.A.)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CHARTIS SEGUROS BRASIL S.A.  "EM APROVAÇÃO" (ANTIGA AIU SEGUR...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDÊNCIA "em aprovação" (antiga SOCIEDADE AUXILIADORA )',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA SOCIEDADE AUXIL...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDÊNCIA "em aprovação" (antiga  SOCIEDADE AUXILIADORA)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA  SOCIEDADE AUXI...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11029',
  'UNIAO DE PREVIEDENCIA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIAO DE PREVIEDENCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21318',
  'RURAL CAPITALIZAÇÃO S/A',
  '710',
  'RURAL',
  'RURAL | RURAL CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('22926',
  'NOSSA CAIXA CAPITALIZAÇÃO S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NOSSA CAIXA CAPITALIZAÇÃO S.A.',
  'CAIXA SEGURIDADE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Nossa Caixa legado',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Média',
  'Banco Nossa Caixa é legado histórico; não tratar como Caixa Econômica. Validar período societário.',
  'SIM'),
 ('26026',
  'LIDERANÇA CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LIDERANÇA CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('27995',
  'HSBC CAPITALIZAÇÃO (BRASIL) S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HSBC CAPITALIZAÇÃO (BRASIL) S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('4120',
  '88i SEGURADORA DIGITAL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | 88I SEGURADORA DIGITAL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10138',
  'BAMÉRCIO S/A PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BAMÉRCIO S/A PREVIDÊNCIA PRIVADA',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('21491',
  'CAPEMISA CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CAPEMISA CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75922',
  'NMB BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NMB BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5177',
  'ALLIANZ SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ALLIANZ SEGUROS S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('40207',
  'ALLIANZ GLOBAL CORPORATE & SPECIALTY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALLIANZ GLOBAL CORPORATE & SPECIALTY',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('46761',
  'Federal Insurance Company',
  '221',
  'CHUBB',
  'CHUBB | FEDERAL INSURANCE COMPANY',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('77721',
  'Benfield do Brasil Corretora de Resseguros Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BENFIELD DO BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4561',
  'Mapfre Nossa Caixa Vida e Previdencia S.A',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE NOSSA CAIXA VIDA E PREVIDENCIA S.A',
  'CAIXA SEGURIDADE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Média',
  'Contém Mapfre Nossa Caixa; tratar como legado Mapfre/BB e validar período.',
  'SIM'),
 ('5096',
  'UNIBANCO AIG VIDA E PREVIDÊNCIA S/A - EM APROVAÇÃO (PHENIX SEGURADORA S/A)',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO AIG VIDA E PREVIDÊNCIA S/A - EM APROVAÇÃO (PHENIX...',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6211',
  'Aliança do Brasil Seguros S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ALIANÇA DO BRASIL SEGUROS S.A',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('24813',
  'APLICAP CAPITALIZAÇÃO S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | APLICAP CAPITALIZAÇÃO S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('41432',
  'Odyssey Reinsurance Company',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ODYSSEY REINSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2054',
  'KAPAM SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | KAPAM SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('41262',
  'SCOR SE',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SCOR SE',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('76503',
  'Howden Re Corretora de Resseguros Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HOWDEN RE CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('40096',
  'STARSTONE SPECIALITY INSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | STARSTONE SPECIALITY INSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4383',
  '180 Seguros S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | 180 SEGUROS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4740',
  'KIRTON VIDA E PREVIDÊNCIA S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | KIRTON VIDA E PREVIDÊNCIA S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('1511',
  'LIGGÊRO SEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LIGGÊRO SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4189',
  'PO SEGURADORA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PO SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6131',
  'COMPANHIA MUTUAL DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPANHIA MUTUAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6173',
  'INVESTPREV SEGUROS E PREVIDÊNCIA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | INVESTPREV SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('4561',
  'BRASILPREV NOSSO FUTURO SEGUROS E PREVIDÊNCIA S/A "em aprovação"',
  '78',
  'BRASIL',
  'BRASIL | BRASILPREV NOSSO FUTURO SEGUROS E PREVIDÊNCIA S/A "EM APROVAÇÃO"',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6955',
  'COMPANHIA DE SEGUROS GRALHA AZUL',
  '35',
  'ITAÚ',
  'ITAÚ | COMPANHIA DE SEGUROS GRALHA AZUL',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('38741',
  'AIG RESSEGUROS BRASIL S.A. (Antiga CHARTIS RESSEGUROS BRASIL S.A.)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AIG RESSEGUROS BRASIL S.A. (ANTIGA CHARTIS RESSEGUROS BRASIL ...',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('1929',
  'HSBC SEGUROS DE AUTOMÓVEIS E BENS (BRASIL) S.A.',
  '1204',
  'HSBC',
  'HSBC | HSBC SEGUROS DE AUTOMÓVEIS E BENS (BRASIL) S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('3298',
  'Mapfre Vera Cruz Previdência S.A. "em aprovação"',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ PREVIDÊNCIA S.A. "EM APROVAÇÃO"',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('3328',
  'Coface do Brasil Seguros de Crédito Interno S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COFACE DO BRASIL SEGUROS DE CRÉDITO INTERNO S/A',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('2763',
  'CESCE Brasil Seguros de Crédito S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CESCE BRASIL SEGUROS DE CRÉDITO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2143',
  'ASSURANT SEGURADORA SA (Nationwide Segur',
  '1213',
  'NATIONWIDE',
  'NATIONWIDE | ASSURANT SEGURADORA SA (NATIONWIDE SEGUR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NATIONWIDE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5096',
  'CIA DE SEGS MAR TER PHENIX DE P.ALEGRE',
  '400',
  'PHENIX DE PORTO ALEGRE',
  'PHENIX DE PORTO ALEGRE | CIA DE SEGS MAR TER PHENIX DE P.ALEGRE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'PHENIX DE PORTO ALEGRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5185',
  'LIBERTY SEGUROS S/A " em aprovação " ( antiga Liberty Paulista Seguros S/A )',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | LIBERTY SEGUROS S/A " EM APROVAÇÃO " ( ANTIGA LIBERTY PAULISTA...',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5282',
  'PRUDENTIAL DO BRASIL SEGUROS DE VIDA S.A',
  '27',
  'BRADESCO',
  'BRADESCO | PRUDENTIAL DO BRASIL SEGUROS DE VIDA S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5282',
  'PRUDENTIAL-BRADESCO SEGUROS S.A',
  '27',
  'BRADESCO',
  'BRADESCO | PRUDENTIAL-BRADESCO SEGUROS S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5487',
  'SANTANDER BRASIL SEGUROS S/A.',
  '370',
  'NOROESTE',
  'NOROESTE | SANTANDER BRASIL SEGUROS S/A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5614',
  'SUL AMÉRICA SANTA CRUZ SEGUROS S/A',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SANTA CRUZ SEGUROS S/A',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5614',
  'SUL AMÉRICA SANTA CRUZ SEGUROS  S/A',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SANTA CRUZ SEGUROS  S/A',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5649',
  'CGU COMPANHIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CGU COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5665',
  'MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A. (EM APROVAÇÃO - ANTIGA VERA CRUZ VID...',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A. (EM APROVAÇÃO - ANTIGA VERA...',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5908',
  'GENERALI DO BRASIL CIA NAC DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | GENERALI DO BRASIL CIA NAC DE SEGUROS',
  'GENERALI',
  'GENERALI',
  'Generali',
  'Seguradora global multilinha',
  'Generali',
  'Alta',
  None,
  'NÃO'),
 ('5908',
  'GENERALI DO BRASIL CIA NACIONAL DE SEGUR',
  '213',
  'GENERALLI',
  'GENERALLI | GENERALI DO BRASIL CIA NACIONAL DE SEGUR',
  'GENERALI',
  'GENERALI',
  'Generali',
  'Seguradora global multilinha',
  'Generali',
  'Alta',
  None,
  'NÃO'),
 ('5932',
  'ALLIANZ - BRADESCO SEGUROS S.A',
  '27',
  'BRADESCO',
  'BRADESCO | ALLIANZ - BRADESCO SEGUROS S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5941',
  'SEGURADORA BMC S/A',
  '1058',
  'BMC',
  'BMC | SEGURADORA BMC S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BMC',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6017',
  'CENTAURO SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CENTAURO SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6131',
  'COMPANHIA MUTUAL DE SEGUROS',
  '1139',
  'MONTEJUS',
  'MONTEJUS | COMPANHIA MUTUAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MONTEJUS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6220',
  'SUL AMÉRICA AETNA SEG.DE VIDA E PREV.S.A',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA AETNA SEG.DE VIDA E PREV.S.A',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6271',
  'SUDAMERIS VIDA E PREVIDENCIA S/A"em aprovação"(antiga SUDAMERIS GENERALI CIA....',
  '86',
  'REAL',
  'REAL | SUDAMERIS VIDA E PREVIDENCIA S/A"EM APROVAÇÃO"(ANTIGA SUDAMERIS GENERA...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'IH CIA. DE SEGUROS E PREVIDÊNCIA',
  '825',
  'ICATU',
  'ICATU | IH CIA. DE SEGUROS E PREVIDÊNCIA',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'PACTUAL PREVID-NCIA E SEGUROS S/A',
  '1082',
  'BRASILEIRA DE FIANCAS',
  'BRASILEIRA DE FIANCAS | PACTUAL PREVID-NCIA E SEGUROS S/A',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVIDÊNCIA  S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | AGF VIDA E PREVIDÊNCIA  S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'ITAUPREV VIDA E PREVIDÊNCIA S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAUPREV VIDA E PREVIDÊNCIA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6408',
  'CIGNA COMPANHIA DE SEGUROS',
  '116',
  'CIGNA',
  'CIGNA | CIGNA COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CIGNA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6424',
  'SDB CIA SEGUROS GERAIS',
  '469',
  'SDB',
  'SDB | SDB CIA SEGUROS GERAIS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SDB',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6467',
  'Alfa Seguros e Previdência S.A.',
  '1192',
  'ALFA',
  'ALFA | ALFA SEGUROS E PREVIDÊNCIA S.A.',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('6467',
  'Alfa Seguradora S.A."em aprovação"(antig',
  '1192',
  'ALFA',
  'ALFA | ALFA SEGURADORA S.A."EM APROVAÇÃO"(ANTIG',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('6548',
  'CARDIF DO BRASIL SEGUROS E PREVIDENCIA S',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CARDIF DO BRASIL SEGUROS E PREVIDENCIA S',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('6548',
  'CARDIF DO BRASIL SEGUROS E PREVIDENCIA S',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CARDIF DO BRASIL SEGUROS E PREVIDENCIA S',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('6564',
  'Santander Brasil Seguros S/A',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER BRASIL SEGUROS S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6602',
  'Mitsui Sumitomo Seguros S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MITSUI SUMITOMO SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6734',
  'AIG BRASIL COMPANHIA DE SEGUROS',
  '108',
  'INTERAMERICANA',
  'INTERAMERICANA | AIG BRASIL COMPANHIA DE SEGUROS',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('6777',
  'APS SEGURADORA S.A.',
  '1221',
  'SULINA',
  'SULINA | APS SEGURADORA S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6831',
  'SINAF PREVIDENCIAL CIA DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SINAF PREVIDENCIAL CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10031',
  'PECÚLIO ABRAHAM LINCOLN - AMAL',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PECÚLIO ABRAHAM LINCOLN - AMAL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10031',
  'PECÚLIO ABRAHAM LINCOLN - AMAL',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PECÚLIO ABRAHAM LINCOLN - AMAL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10332',
  'COIFA PECULIOS E PENSOES',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COIFA PECULIOS E PENSOES',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10405',
  'Empresarial de Previdencia Privada',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EMPRESARIAL DE PREVIDENCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11045',
  'BRASILPREV SEGUROS E PREV.S/A(BRASILPREV',
  '78',
  'BRASIL',
  'BRASIL | BRASILPREV SEGUROS E PREV.S/A(BRASILPREV',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('21482',
  'UNIBANCO CIA DE CAPITALIZAÇÃO',
  '1191',
  'UNIBANCO/AIG',
  'UNIBANCO/AIG | UNIBANCO CIA DE CAPITALIZAÇÃO',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('70076',
  'MEXBRIT BRASIL CONSULTORIA DE SEGUROS E RESSEGUROS LTDA.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MEXBRIT BRASIL CONSULTORIA DE SEGUROS E RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71498',
  'Securitas União Corretora de Resseguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SECURITAS UNIÃO CORRETORA DE RESSEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76554',
  'ALBATROZ CORRETORA DE RESSEGUROS LTDA',
  '78',
  'BRASIL',
  'BRASIL | ALBATROZ CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BRASIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4740',
  'HSBC VIDA E PREVIDÊNCIA (Brasil) S.A.',
  '1204',
  'HSBC',
  'HSBC | HSBC VIDA E PREVIDÊNCIA (BRASIL) S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5941',
  'QBE BRASIL SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | QBE BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75434',
  'WILLIS CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | WILLIS CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6416',
  'YASUDA SEGUROS S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | YASUDA SEGUROS S.A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6696',
  'SUL AMÉRICA COMPANHIA DE SEGUROS GERAIS',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA COMPANHIA DE SEGUROS GERAIS',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('10707',
  'MBM Previdência Privada',
  '1121',
  'MBM',
  'MBM | MBM PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MBM',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1627',
  'Safra Seguros Gerais S.A.',
  '345',
  'SAFRA',
  'SAFRA | SAFRA SEGUROS GERAIS S.A.',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('48879',
  'General Reinsurance AG',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GENERAL REINSURANCE AG',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2461',
  'Austral Seguradora S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AUSTRAL SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6351',
  'METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('6513',
  'ACE SEGURADORA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ACE SEGURADORA S.A.',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('8737',
  'AIG SEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AIG SEGUROS BRASIL S.A.',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('28878',
  'PORTO SEGURO CAPITALIZAÇÃO S.A.',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO CAPITALIZAÇÃO S.A.',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('71412',
  'UNIVERSAL RE CORRETORES DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UNIVERSAL RE CORRETORES DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71498',
  'Arthur J. Gallagher Brasil Corretora de Resseguros Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARTHUR J. GALLAGHER BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20141',
  'BRASILCAP CAPITALIZAÇÃO S/A',
  '78',
  'BRASIL',
  'BRASIL | BRASILCAP CAPITALIZAÇÃO S/A',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilcap',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('42951',
  'TRANSAMERICA INTERNATIONAL RE (BERMUDA) LTD',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TRANSAMERICA INTERNATIONAL RE (BERMUDA) LTD',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2739',
  'AGÊNCIA BRASILEIRA GESTORA DE FUNDOS GARANTIDORES E GARANTIAS S.A. \x96 ABGF',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AGÊNCIA BRASILEIRA GESTORA DE FUNDOS GARANTIDORES E GARANTIAS...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6033',
  'PORTO SEGURO VIDA E PREVIDÊNCIA S.A.',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO VIDA E PREVIDÊNCIA S.A.',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('6238',
  'MAPFRE SEGUROS GERAIS S.A.',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | MAPFRE SEGUROS GERAIS S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('1589',
  'SANTANDER AUTO S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SANTANDER AUTO S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('2305',
  'BTG Pactual Seguros S.A.',
  '1201',
  'BANCO PACTUAL',
  'BANCO PACTUAL | BTG PACTUAL SEGUROS S.A.',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('2682',
  'BTG Pactual Vida e Previdência S.A.',
  '1201',
  'BANCO PACTUAL',
  'BANCO PACTUAL | BTG PACTUAL VIDA E PREVIDÊNCIA S.A.',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('79006',
  'LOCKTON RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LOCKTON RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75957',
  'INTER CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | INTER CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3263',
  'PIER SEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PIER SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75434',
  'WILLIS CORRETORA DE RESSEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | WILLIS CORRETORA DE RESSEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3948',
  'SEGURADORA SA INFINITE',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA SA INFINITE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1503',
  'FM SEGUROS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FM SEGUROS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2241',
  'Justos Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | JUSTOS SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('46507',
  'AXIS RE PLC',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AXIS RE PLC',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3727',
  'TRAVELERS SEGUROS BRASIL S/A (em aprovação da SUSEP - antiga J.MALUCELLI SEGU...',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | TRAVELERS SEGUROS BRASIL S/A (EM APROVAÇÃO DA SUSEP - ANTIGA J...',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('3166',
  'CITIINSURANCE DO BRASIL VIDA E PREV S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CITIINSURANCE DO BRASIL VIDA E PREV S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5185',
  'Liberty Paulista Seguros S/A',
  '167',
  'PAULISTA',
  'PAULISTA | LIBERTY PAULISTA SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5355',
  'UAP SEGUROS BRASIL S/A.',
  '299',
  'UAP',
  'UAP | UAP SEGUROS BRASIL S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('21318',
  'RURAL CAPITALIZAÇÃO S.A',
  '710',
  'RURAL',
  'RURAL | RURAL CAPITALIZAÇÃO S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5665',
  'Mapfre Vera Cruz Vida e Previdência S.A',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5959',
  'SEGURANÇA CIA DE SEGUROS E PREVIDENCIA',
  '1066',
  'SEGURANCA',
  'SEGURANCA | SEGURANÇA CIA DE SEGUROS E PREVIDENCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SEGURANCA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6157',
  'M.B.VIDA E PREV.S/A "EM APROVAÇÃO" (ANTI',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | M.B.VIDA E PREV.S/A "EM APROVAÇÃO" (ANTI',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6203',
  'ZURICH BRASIL SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ZURICH BRASIL SEGUROS S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6211',
  'Santa Catarina Seguros e Previdência S.A',
  '1220',
  'BANCO EST STA CATARINA',
  'BANCO EST STA CATARINA | SANTA CATARINA SEGUROS E PREVIDÊNCIA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BANCO EST STA CATARINA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6378',
  'DINÂMICA SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | DINÂMICA SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6416',
  'YASUDA SEGUROS S.A',
  '1224',
  'YASUDA',
  'YASUDA | YASUDA SEGUROS S.A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6483',
  'XL Insurance Brazil Seg S/A"em aprovação',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | XL INSURANCE BRAZIL SEG S/A"EM APROVAÇÃO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6530',
  'COMBINED SEGUROS BRASIL S/A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMBINED SEGUROS BRASIL S/A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6548',
  'CARDIF DO BRASIL VIDA E PREVIDÊNCIA S/A "EM APROVAÇÃO" (CARDIF DO BRASIL SEGU...',
  '1203',
  'CARDIF',
  'CARDIF | CARDIF DO BRASIL VIDA E PREVIDÊNCIA S/A "EM APROVAÇÃO" (CARDIF DO BR...',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('6726',
  'MULTIPLIC SEGURADORA S/A.',
  '299',
  'UAP',
  'UAP | MULTIPLIC SEGURADORA S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('6807',
  'ABSOLUTA SEGUROS S.A. (Em aprovação, ADR',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ABSOLUTA SEGUROS S.A. (EM APROVAÇÃO, ADR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6840',
  'Prever SA Seguros e Previdência',
  '701',
  'PREVER',
  'PREVER | PREVER SA SEGUROS E PREVIDÊNCIA',
  'OUTROS',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'SIM'),
 ('10014',
  'ACVAT- PREVIDENCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ACVAT- PREVIDENCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10031',
  'VIVER PREVIDÊNCIA "em aprovação" (antiga PECÚLIO ABRAHAM LINCOLN - AMAL)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIVER PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA PECÚLIO ABRAHAM LINCO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10669',
  'RS PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | RS PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDÊNCIA " em aprovação" (antiga SOCIEDADE AUXILIADORA)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDÊNCIA " EM APROVAÇÃO" (ANTIGA SOCIEDADE AUXI...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11029',
  'SUCV UNIÃO DE PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SUCV UNIÃO DE PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20273',
  'Valor Capitalização S.A.(Santos Capitali',
  '1112',
  'SANTOS',
  'SANTOS | VALOR CAPITALIZAÇÃO S.A.(SANTOS CAPITALI',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20451',
  'MEGACAP CAPITALIZAÇÃO S/A "EM APROVAÇÃO" (ANTIGA GLOBAL CAPITALIZACAO S/A)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MEGACAP CAPITALIZAÇÃO S/A "EM APROVAÇÃO" (ANTIGA GLOBAL CAPIT...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20621',
  'BANDEIRANTES S/A CAPITALIZAÇÃO',
  '426',
  'TREVO',
  'TREVO | BANDEIRANTES S/A CAPITALIZAÇÃO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'TREVO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21334',
  'ICATU CAPITALIZAÇÃO S.A." em aprovação" (antiga ICATU HARTFORD CAPITALIZAÇÃO ...',
  '825',
  'ICATU',
  'ICATU | ICATU CAPITALIZAÇÃO S.A." EM APROVAÇÃO" (ANTIGA ICATU HARTFORD CAPITA...',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('45926',
  'ARIEL REINSURANCE COMPANY LTD',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARIEL REINSURANCE COMPANY LTD',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70076',
  'MEXBRIT BRASIL CONSULTORIA DE SEGUROS E RESSEGUROS LTDA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MEXBRIT BRASIL CONSULTORIA DE SEGUROS E RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3794',
  'dasseg',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | DASSEG',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2895',
  'Alfa Previdência e Vida S.A.',
  '1192',
  'ALFA',
  'ALFA | ALFA PREVIDÊNCIA E VIDA S.A.',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('5274',
  'Banestes Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BANESTES SEGUROS S/A',
  'OUTROS',
  'BANESTES',
  'Banestes Seguros',
  'Bancassurance / banco regional',
  'Banestes',
  'Alta',
  None,
  'SIM'),
 ('5631',
  'CAIXA SEGURADORA S/A',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXA SEGURADORA S/A',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('4707',
  'BRASILPREV SEGUROS E PREVIDÊNCIA S/A',
  '78',
  'BRASIL',
  'BRASIL | BRASILPREV SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6220',
  'Sul América Seguros de Pessoas e Previdência S.A.',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SEGUROS DE PESSOAS E PREVIDÊNCIA S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('23540',
  'APLUB Capitalização S. A.',
  '990',
  'APLUB',
  'APLUB | APLUB CAPITALIZAÇÃO S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'APLUB',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6912',
  'ITAU BMG SEGURADORA SA "EM APROVAÇAO"',
  '35',
  'ITAÚ',
  'ITAÚ | ITAU BMG SEGURADORA SA "EM APROVAÇAO"',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5070',
  'Santander Seguros S/A (em aprov. Santand',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER SEGUROS S/A (EM APROV. SANTAND',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('3182',
  'ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A."em aprovação" (antiga UASEG SEGUROS)',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A."EM APROVAÇÃO" (ANTIGA UASEG SEG...',
  'ITAÚ UNIBANCO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'SIM'),
 ('11070',
  'União Previdenciária Cometa do Brasil',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UNIÃO PREVIDENCIÁRIA COMETA DO BRASIL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('41491',
  'HANNOVER RUCKVERSICHERUNG AG',
  '1210',
  'HANNOVER',
  'HANNOVER | HANNOVER RUCKVERSICHERUNG AG',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('4561',
  'Nossa Caixa Seguros e Previdência S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NOSSA CAIXA SEGUROS E PREVIDÊNCIA S.A.',
  'CAIXA SEGURIDADE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Nossa Caixa legado',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Média',
  'Banco Nossa Caixa é legado histórico; não tratar como Caixa Econômica. Validar período societário.',
  'SIM'),
 ('6190',
  'TOKIO MARINE SEGURADORA S.A.',
  '1222',
  'TOKIO MARINE',
  'TOKIO MARINE | TOKIO MARINE SEGURADORA S.A.',
  'TOKIO MARINE',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'NÃO'),
 ('1741',
  'BMG SEGUROS SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BMG SEGUROS SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6238',
  'MAPFRE SEGUROS GERAIS S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE SEGUROS GERAIS S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5690',
  'Companhia Excelsior de Seguros',
  '612',
  'EXCELSIOR',
  'EXCELSIOR | COMPANHIA EXCELSIOR DE SEGUROS',
  'OUTROS',
  'EXCELSIOR',
  'Excelsior',
  'Seguradora nacional independente',
  'Excelsior',
  'Alta',
  None,
  'SIM'),
 ('26158',
  'KIRTON CAPITALIZAÇÃO S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | KIRTON CAPITALIZAÇÃO S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('1741',
  'DAYCOVAL SEGUROS SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | DAYCOVAL SEGUROS SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6301',
  'RIO GRANDE SEGUROS E PREVIDÊNCIA S.A.',
  '825',
  'ICATU',
  'ICATU | RIO GRANDE SEGUROS E PREVIDÊNCIA S.A.',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6921',
  'KOVR SEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | KOVR SEGURADORA S.A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('5941',
  'ZURICH BRASIL COMPANHIA DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ZURICH BRASIL COMPANHIA DE SEGUROS',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6211',
  'Aliança do Brasil Seguros S.A',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | ALIANÇA DO BRASIL SEGUROS S.A',
  'MAPFRE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('17400',
  'Evidence Previdência',
  '1200',
  'SANTANDER',
  'SANTANDER | EVIDENCE PREVIDÊNCIA',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('3727',
  'FAIRWAY SEGUROS S.A. (nova denominação de Travelers Seguros Brasil S.A.)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FAIRWAY SEGUROS S.A. (NOVA DENOMINAÇÃO DE TRAVELERS SEGUROS B...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('25585',
  'CNP CAPITALIZAÇÃO S/A',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CNP CAPITALIZAÇÃO S/A',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('1481',
  'SOMPO CONSUMER SEGURADORA SA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOMPO CONSUMER SEGURADORA SA',
  'OUTROS',
  'HDI / TALANX',
  'HDI / Sompo Consumer',
  'Seguradora global multilinha',
  'Sompo Consumer adquirida pela HDI',
  'Alta',
  'Sompo Consumer/varejo; não confundir com Sompo corporativo.',
  'SIM'),
 ('6921',
  'Investprev Seguradora S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | INVESTPREV SEGURADORA S.A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('4359',
  'Euler do Brasil Seguros S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER DO BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6181',
  'Brasilveículos Companhia de Seguros',
  '1212',
  'MAPFRE',
  'MAPFRE | BRASILVEÍCULOS COMPANHIA DE SEGUROS',
  'MAPFRE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6041',
  'PARANA COMPANHIA DE SEGUROS',
  '35',
  'ITAÚ',
  'ITAÚ | PARANA COMPANHIA DE SEGUROS',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('3816',
  'Real Seguros Vida e Previdência S.A. (nova denomin Social Real Tokio Marine V...',
  '1200',
  'SANTANDER',
  'SANTANDER | REAL SEGUROS VIDA E PREVIDÊNCIA S.A. (NOVA DENOMIN SOCIAL REAL TO...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('1571',
  'HDI GLOBAL SEGUROS S.A. - "em aprovação" (antiga HDI-Gerling Seguros Industri...',
  '1230',
  'TALANX AG',
  'TALANX AG | HDI GLOBAL SEGUROS S.A. - "EM APROVAÇÃO" (ANTIGA HDI-GERLING SEGU...',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('1554',
  'EQUATORIAL VIDA E PREVIDENCIA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EQUATORIAL VIDA E PREVIDENCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5096',
  'UNIBANCO VIDA E PREVIDÊNCIA S/A "Em aprovação" (antiga Unibanco AIG Vida e Pr...',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO VIDA E PREVIDÊNCIA S/A "EM APROVAÇÃO" (ANTIGA UNI...',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5495',
  'Cia. Seguros Minas-Brasil',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | CIA. SEGUROS MINAS-BRASIL',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('5525',
  'INTERBRAZIL SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | INTERBRAZIL SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5649',
  'GENERAL ACCIDENT CIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GENERAL ACCIDENT CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5711',
  'BRADESCO  SAUDE S.A',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO  SAUDE S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5991',
  'SEGURADORA BRASILEIRA DE FIANÇAS S/A',
  '1082',
  'BRASILEIRA DE FIANCAS',
  'BRASILEIRA DE FIANCAS | SEGURADORA BRASILEIRA DE FIANÇAS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BRASILEIRA DE FIANCAS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6050',
  'MAXLIFE SEGURADORA DO BRASIL S/A  (EM AP',
  '1023',
  'NOBRE',
  'NOBRE | MAXLIFE SEGURADORA DO BRASIL S/A  (EM AP',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NOBRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6050',
  'MAXMED SEGURADORA S/A ( MAXLIFE EM APROV',
  '1023',
  'NOBRE',
  'NOBRE | MAXMED SEGURADORA S/A ( MAXLIFE EM APROV',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NOBRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6106',
  'HSBC SEGUROS (BRASIL)  S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HSBC SEGUROS (BRASIL)  S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6122',
  'CIGNA SEGURADORA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CIGNA SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6220',
  'Sul América Seguros de Vida e Previdênci',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SEGUROS DE VIDA E PREVIDÊNCI',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6246',
  'Sul America Aetna Seguros e Previdencia',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMERICA AETNA SEGUROS E PREVIDENCIA',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6335',
  'Seguradora Brasileira de Crédito à Expor',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA BRASILEIRA DE CRÉDITO À EXPOR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6572',
  'HANNOVER INTERNATIONAL SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HANNOVER INTERNATIONAL SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6572',
  'HANNOVER INTERNATIONAL SEGUROS S.A.',
  '1210',
  'HANNOVER',
  'HANNOVER | HANNOVER INTERNATIONAL SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6653',
  'PANAMERICANA DE SEGUROS S-A',
  '582',
  'PANAMERICANA',
  'PANAMERICANA | PANAMERICANA DE SEGUROS S-A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'PANAMERICANA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6688',
  'COSESP - CIA DE SEGUROS DO ESTADO DE SÃO',
  '159',
  'COSESP',
  'COSESP | COSESP - CIA DE SEGUROS DO ESTADO DE SÃO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'COSESP',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6726',
  'RIO BRANCO SEGURADORA S/A.',
  '1195',
  'AXA',
  'AXA | RIO BRANCO SEGURADORA S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('6769',
  'United SBF Seguros S/A',
  '1171',
  'UNITED',
  'UNITED | UNITED SBF SEGUROS S/A',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('6807',
  'ADRESS SEGUROS E PREVIDÊNCIA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ADRESS SEGUROS E PREVIDÊNCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6858',
  'SEGURADORA ROMA S/A',
  '752',
  'ROMA',
  'ROMA | SEGURADORA ROMA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'ROMA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6858',
  'MARES - MAPFRE RISCOS ESPECIAIS SEGURADORA S.A. "em aprovação" (antiga Segura...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MARES - MAPFRE RISCOS ESPECIAIS SEGURADORA S.A. "EM APROVAÇÃO"...',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6980',
  'NOTRE DAME SEGURADORA S/A',
  '841',
  'NOTRE DAME',
  'NOTRE DAME | NOTRE DAME SEGURADORA S/A',
  'NOTRE DAME INTERMÉDICA',
  'HAPVIDA NOTREDAME',
  'NotreDame Intermédica / Golden Cross',
  'Saúde/operadora',
  'NotreDame/Golden Cross',
  'Média',
  'Validar se entidade regulada SUSEP é seguradora legada ou operação de saúde/odonto.',
  'NÃO'),
 ('8737',
  'AIG BRASIL COMPANHIA DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AIG BRASIL COMPANHIA DE SEGUROS',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('10413',
  'FAMILIA BANDEIRANTE PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FAMILIA BANDEIRANTE PREVIDÊNCIA PRIVADA',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('10634',
  'Mongeral Previdência Privada',
  '825',
  'ICATU',
  'ICATU | MONGERAL PREVIDÊNCIA PRIVADA',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('10847',
  'PREVIMIL SOC DE PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PREVIMIL SOC DE PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70076',
  'MEXBRIT CONSULTORIA DE SEGUROS E RESSEGUROS LTDA.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MEXBRIT CONSULTORIA DE SEGUROS E RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71412',
  'ADAMS & PORTER CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ADAMS & PORTER CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75574',
  'SAVE-RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SAVE-RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77895',
  'GALCORR CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GALCORR CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5720',
  'Sompo Seguros S.A. (em aprovação)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOMPO SEGUROS S.A. (EM APROVAÇÃO)',
  'OUTROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'SIM'),
 ('3182',
  'UASEG SEGUROS S/A.',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UASEG SEGUROS S/A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5665',
  'MAPFRE VIDA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE VIDA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6858',
  'MAPFRE AFFINITY SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE AFFINITY SEGURADORA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('11134',
  'ASPECIR PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ASPECIR PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6335',
  'Seguradora Brasileira de Crédito à Expor',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEGURADORA BRASILEIRA DE CRÉDITO À EXPOR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6572',
  'HDI SEGUROS S.A.',
  '1230',
  'TALANX AG',
  'TALANX AG | HDI SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('48747',
  'SWISS REINSURANCE AMERICA CORPORATION',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SWISS REINSURANCE AMERICA CORPORATION',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('4634',
  'JAVA NORDESTE SEGUROS S/A',
  '1227',
  'SINAF',
  'SINAF | JAVA NORDESTE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SINAF',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3182',
  'ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A.',
  'ITAÚ UNIBANCO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'SIM'),
 ('75001',
  'CATALYST RE CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CATALYST RE CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78778',
  'ORYPABA RIO ADMINISTRAÇÃO E CORRETAGEM DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ORYPABA RIO ADMINISTRAÇÃO E CORRETAGEM DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3727',
  'J.MALUCELLI SEGUROS S/A (em aprovação) anterior J.MALUCELLI SEGURADORA DE CRE...',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | J.MALUCELLI SEGUROS S/A (EM APROVAÇÃO) ANTERIOR J.MALUCELLI SE...',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('25585',
  'CAIXA CAPITALIZAÇÃO S/A',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXA CAPITALIZAÇÃO S/A',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('47708',
  'CATLIN INSURANCE COMPANY (UK) LTD.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CATLIN INSURANCE COMPANY (UK) LTD.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70092',
  'LOCKTON RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LOCKTON RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74934',
  'BTG Pactual Corretora de Resseguros Ltda.',
  '1201',
  'BANCO PACTUAL',
  'BANCO PACTUAL | BTG PACTUAL CORRETORA DE RESSEGUROS LTDA.',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('76503',
  'Pluris Re Corretora de Resseguros Ltda.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PLURIS RE CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76767',
  'Som.us do Brasil Corretora de Resseguros Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SOM.US DO BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1546',
  'BP SEGURADORA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BP SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2577',
  'COBUCCIO SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COBUCCIO SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4421',
  'XS2 VIDA E PREVIDÊNCIA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XS2 VIDA E PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('45951',
  'ATRADIUS REINSURANCE DESIGNATED ACTIVITY COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ATRADIUS REINSURANCE DESIGNATED ACTIVITY COMPANY',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('24872',
  'XS4 CAPITALIZAÇÃO S.A.',
  '825',
  'ICATU',
  'ICATU | XS4 CAPITALIZAÇÃO S.A.',
  'ICATU',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'SIM'),
 ('3387',
  'Angelus Seguros S/A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ANGELUS SEGUROS S/A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4537',
  'DARWIN SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | DARWIN SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4936',
  'COMPANHIA CAPITAL DE SEGUROS - MICROSSEGURADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMPANHIA CAPITAL DE SEGUROS - MICROSSEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5665',
  'MAPFRE VIDA S.A.',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | MAPFRE VIDA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('3786',
  'COMPROSEGUROS SEGURADORA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPROSEGUROS SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10014',
  'ACVAT- PREVIDENCIA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ACVAT- PREVIDENCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2062',
  'Luizaseg Seguros S/A "em aprovação" (Cardif do Brasil Seguros Gerais S/A "ant...',
  '1203',
  'CARDIF',
  'CARDIF | LUIZASEG SEGUROS S/A "EM APROVAÇÃO" (CARDIF DO BRASIL SEGUROS GERAIS...',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('4359',
  'Euler Hermes Seg.Créd.S/A"em aprov" (ant',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER HERMES SEG.CRÉD.S/A"EM APROV" (ANT',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('17400',
  'EVIDENCE PREVIDÊNCIA S.A.',
  '1200',
  'SANTANDER',
  'SANTANDER | EVIDENCE PREVIDÊNCIA S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('38741',
  'CHARTIS RESSEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CHARTIS RESSEGUROS BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4367',
  'IU SEGUROS S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | IU SEGUROS S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('11070',
  'Comprev Previdência S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPREV PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('8737',
  'AIG SEGUROS BRASIL S.A. (antiga CHARTIS SEGUROS BRASIL S.A.)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AIG SEGUROS BRASIL S.A. (ANTIGA CHARTIS SEGUROS BRASIL S.A.)',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('6947',
  'UNIMED SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIMED SEGURADORA S/A',
  'UNIMED',
  'UNIMED',
  'Unimed',
  'Saúde/operadora/cooperativa',
  'Unimed',
  'Alta',
  None,
  'NÃO'),
 ('2763',
  'SEGURADORA DE CRÉDITO DO BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA DE CRÉDITO DO BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5185',
  'LIBERTY SEGUROS S/A',
  '1230',
  'TALANX AG',
  'TALANX AG | LIBERTY SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5185',
  'Liberty Paulista Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | LIBERTY PAULISTA SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5533',
  'FINASA SEGURADORA S/A',
  '94',
  'FINASA',
  'FINASA | FINASA SEGURADORA S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5665',
  'Vera Cruz Vida e Previdência S.A',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | VERA CRUZ VIDA E PREVIDÊNCIA S.A',
  'OUTROS',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'SIM'),
 ('5665',
  'MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A. (EM APROVAÇÃO - ANTIGA VERA CRUZ VID...',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A. (EM APROVAÇÃO - ANTIGA V...',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5878',
  'SULINA SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SULINA SEGURADORA S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6092',
  'NOVO HAMBURGO CIA. DE SEGUROS GERAIS',
  '27',
  'BRADESCO',
  'BRADESCO | NOVO HAMBURGO CIA. DE SEGUROS GERAIS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6190',
  'REAL PREVIDENCIA E SEGUROS S.A.',
  '86',
  'REAL',
  'REAL | REAL PREVIDENCIA E SEGUROS S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6220',
  'SAm Seg.Vd.Prev.SA"em aprov"(antiga SAm',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SAM SEG.VD.PREV.SA"EM APROV"(ANTIGA SAM',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6602',
  'Mitsui Marine & Kyoei Fire Seguros S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MITSUI MARINE & KYOEI FIRE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6602',
  'MITSUI MARINE & KYOEI FIRE SEGUROS S/A',
  '442',
  'CONCÓRDIA',
  'CONCÓRDIA | MITSUI MARINE & KYOEI FIRE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CONCÓRDIA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6769',
  'UBF GARANTIAS & SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UBF GARANTIAS & SEGUROS S/A',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6807',
  'ADRESS SEGUROS E PREVIDÊNCIA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ADRESS SEGUROS E PREVIDÊNCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6939',
  'Seguradora América do Sul S.A.',
  '1215',
  'METROPOLITAN LIFE',
  'METROPOLITAN LIFE | SEGURADORA AMÉRICA DO SUL S.A.',
  'SANTANDER',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'SIM'),
 ('10596',
  'PREVBRAS SOC NAC DE PREV PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVBRAS SOC NAC DE PREV PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10634',
  'Mongeral Previdência Privada',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MONGERAL PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10901',
  'SABEMI PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SABEMI PREVIDÊNCIA PRIVADA',
  'SABEMI',
  'SABEMI',
  'Sabemi',
  'Vida/previdência',
  'Sabemi',
  'Alta',
  None,
  'NÃO'),
 ('11029',
  'SUCV UNIÃO DE PREVIDÊNCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUCV UNIÃO DE PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11029',
  'UNIÃO DE PREVIDÊNCIA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIÃO DE PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11134',
  'ASPECIR PREVIDÊNCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ASPECIR PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23345',
  'ATLANTICA CAPITALIZAÇÃO S/A "EM APROVAÇÃ',
  '27',
  'BRADESCO',
  'BRADESCO | ATLANTICA CAPITALIZAÇÃO S/A "EM APROVAÇÃ',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('26026',
  'LIDERANÇA CAPITALIZAÇÃOS/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LIDERANÇA CAPITALIZAÇÃOS/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('28932',
  'SUL AMÉRICA CAPITALIZAÇÃO S.A. - SULACAP "EM APROVAÇÃO" (ANTIGA SATPAR CAPITA...',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA CAPITALIZAÇÃO S.A. - SULACAP "EM APROVAÇÃO" (ANTIGA...',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('29319',
  'Santander Capitalização S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SANTANDER CAPITALIZAÇÃO S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('70203',
  'BSR Brasil Special Risks Corretora de Resseguros Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BSR BRASIL SPECIAL RISKS CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71498',
  'Arthur J. Gallagher Brasil Corretora de Resseguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARTHUR J. GALLAGHER BRASIL CORRETORA DE RESSEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('73296',
  'SWINGLEHURST BRASIL RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SWINGLEHURST BRASIL RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78778',
  'ORYPABA RIO ADMINISTRAÇÃO E CORRETAGEM DE SEGUROS E RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ORYPABA RIO ADMINISTRAÇÃO E CORRETAGEM DE SEGUROS E RESSEGURO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5355',
  'AZUL COMPANHIA DE SEGUROS GERAIS',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | AZUL COMPANHIA DE SEGUROS GERAIS',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('38741',
  'AIG RESSEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AIG RESSEGUROS BRASIL S.A.',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('5720',
  'Maritima Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MARITIMA SEGUROS S/A',
  'OUTROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'SIM'),
 ('10901',
  'SABEMI PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SABEMI PREVIDÊNCIA PRIVADA',
  'SABEMI',
  'SABEMI',
  'Sabemi',
  'Vida/previdência',
  'Sabemi',
  'Alta',
  None,
  'NÃO'),
 ('10014',
  'ACVAT- PREVIDENCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ACVAT- PREVIDENCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10120',
  'EQUATORIAL PREVIDÊNCIA COMPLEMENTAR',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EQUATORIAL PREVIDÊNCIA COMPLEMENTAR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('41416',
  'FACTORY MUTUAL INSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FACTORY MUTUAL INSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11088',
  'FUTURO PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | FUTURO PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('22683',
  'Zurich Brasil Capitalização S.A.',
  '531',
  'ZURICH',
  'ZURICH | ZURICH BRASIL CAPITALIZAÇÃO S.A.',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('33294',
  'MAPFRE RE DO BRASIL CIA DE RESSEGURO',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE RE DO BRASIL CIA DE RESSEGURO',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6653',
  'Pan Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PAN SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2119',
  'ARUANA SEGUROS S. A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ARUANA SEGUROS S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3727',
  'J.MALUCELLI SEGURADORA DE CREDITO SA',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | J.MALUCELLI SEGURADORA DE CREDITO SA',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('5151',
  'TOKIO MARINE BRASIL SEGURADORA S/A',
  '1222',
  'TOKIO MARINE',
  'TOKIO MARINE | TOKIO MARINE BRASIL SEGURADORA S/A',
  'TOKIO MARINE',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'NÃO'),
 ('5070',
  'Zurich Santander Brasil Seguros e Previdencia S/A (em aprovação) antiga Santa...',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZURICH SANTANDER BRASIL SEGUROS E PREVIDENCIA S/A (EM APROVAÇ...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('3298',
  'Mapfre Seguradora de Garantias e Crédito',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE SEGURADORA DE GARANTIAS E CRÉDITO',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('44938',
  'FINANCIAL ASSURANCE COMPANY LIMITED',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FINANCIAL ASSURANCE COMPANY LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1481',
  'HDI Seguros do Brasil S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HDI SEGUROS DO BRASIL S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('3476',
  'XS3 SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XS3 SEGUROS S.A.',
  'OUTROS',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'SIM'),
 ('2771',
  'SUDASEG SEGURADORA DE DANOS E PESSOAS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUDASEG SEGURADORA DE DANOS E PESSOAS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23540',
  'VIA CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VIA CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4634',
  'PATER SEGUROS S.A.',
  '1227',
  'SINAF',
  'SINAF | PATER SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SINAF',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23540',
  'APLUB CAPITALIZAÇÃO S. A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | APLUB CAPITALIZAÇÃO S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76767',
  'C6 SEG DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | C6 SEG DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1554',
  'EQUATORIAL SEGURADORA S/A - MICROSSEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EQUATORIAL SEGURADORA S/A - MICROSSEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3956',
  'FAM SEGURADORA DE CRÉDITO E GARANTIA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FAM SEGURADORA DE CRÉDITO E GARANTIA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5193',
  'CIA SEGUROS PREVIDENCIA DO SUL',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CIA SEGUROS PREVIDENCIA DO SUL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('41629',
  'SCOR REINSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SCOR REINSURANCE COMPANY',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('6653',
  'Pan Seguros S/A "em aprovação" (antiga Panamericana de Seguros S/A.)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PAN SEGUROS S/A "EM APROVAÇÃO" (ANTIGA PANAMERICANA DE SEGURO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3727',
  'TRAVELERS SEGUROS BRASIL S.A.',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | TRAVELERS SEGUROS BRASIL S.A.',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('77003',
  'BRAZIL RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BRAZIL RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('39764',
  'ALTERRA RESSEGURADORA DO BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALTERRA RESSEGURADORA DO BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2119',
  'ARUANA SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ARUANA SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('44814',
  'Partner Reinsurance Europe Limited',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PARTNER REINSURANCE EUROPE LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4618',
  'ITAÚ UNIBANCO SEGUROS CORPORATIVOS S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ UNIBANCO SEGUROS CORPORATIVOS S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('1929',
  'HDI SEGUROS DE AUTOMÓVEIS E BENS S.A.',
  '1210',
  'HANNOVER',
  'HANNOVER | HDI SEGUROS DE AUTOMÓVEIS E BENS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('4634',
  'JAVA NORDESTE SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | JAVA NORDESTE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('49000',
  'GENERAL INSURANCE CORPORATION OF INDIA',
  '78',
  'BRASIL',
  'BRASIL | GENERAL INSURANCE CORPORATION OF INDIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BRASIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6564',
  'Zurich Santander Brasil Seguros S/A (em aprovação) antiga Santander Brasil Se...',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZURICH SANTANDER BRASIL SEGUROS S/A (EM APROVAÇÃO) ANTIGA SAN...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5169',
  'SAU SEGUROS E PREVIDENCIA S/A',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | SAU SEGUROS E PREVIDENCIA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5185',
  'Companhia Paulista de Seguros',
  '167',
  'PAULISTA',
  'PAULISTA | COMPANHIA PAULISTA DE SEGUROS',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5231',
  'BANCRED SEGURADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BANCRED SEGURADORA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5231',
  'BANCRED SEGURADORA S.A.',
  '892',
  'BANCRED',
  'BANCRED | BANCRED SEGURADORA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5282',
  'PRUDENTIAL- BRADESCO SEGUROS S.A',
  '27',
  'BRADESCO',
  'BRADESCO | PRUDENTIAL- BRADESCO SEGUROS S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5428',
  'Martinelli Seguradora S/A',
  '931',
  'MONAVAL',
  'MONAVAL | MARTINELLI SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MONAVAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5444',
  'BRADESCO SEGUROS S..A',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO SEGUROS S..A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5495',
  'Cia. Seguros Minas-Brasil',
  '1207',
  'BANCO MERCANTIL DO BRASIL',
  'BANCO MERCANTIL DO BRASIL | CIA. SEGUROS MINAS-BRASIL',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('5975',
  'BCN Seguradora S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | BCN SEGURADORA S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5991',
  'Seguradora Brasileira Rural S/A.(em aprovação) anteriormente denominada Segur...',
  '1229',
  'UBF',
  'UBF | SEGURADORA BRASILEIRA RURAL S/A.(EM APROVAÇÃO) ANTERIORMENTE DENOMINADA...',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6157',
  'M.B.SEG.VIDA E PREV.S/A "EM APROV."(ANTI',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | M.B.SEG.VIDA E PREV.S/A "EM APROV."(ANTI',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6157',
  'MINAS BRASIL VEÍCULOS SEGURADORA S/A',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | MINAS BRASIL VEÍCULOS SEGURADORA S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6190',
  'REAL SEGUROS S.A."em aprovação"(antiga R',
  '86',
  'REAL',
  'REAL | REAL SEGUROS S.A."EM APROVAÇÃO"(ANTIGA R',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6190',
  'REAL SEGUROS S.A."em aprovação"(antiga Real Previd e Seguros',
  '86',
  'REAL',
  'REAL | REAL SEGUROS S.A."EM APROVAÇÃO"(ANTIGA REAL PREVID E SEGUROS',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6238',
  'VERA CRUZ SEGURADORA S/A',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | VERA CRUZ SEGURADORA S/A',
  'OUTROS',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'SIM'),
 ('6238',
  'MAPFRE VERA CRUZ SEGURADORA S/A (EM APROVAÇÃO - ANTIGA VERA CRUZ SEGURADORA S...',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ SEGURADORA S/A (EM APROVAÇÃO - ANTIGA VERA CRUZ SEG...',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6343',
  'SOL DE SEGUROS S/A',
  '132',
  'FEDERAL',
  'FEDERAL | SOL DE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'FEDERAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6408',
  'EXCEL CIGNA SEGURADORA S/A',
  '116',
  'CIGNA',
  'CIGNA | EXCEL CIGNA SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CIGNA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6467',
  'Alfa Seguros e Previdência S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALFA SEGUROS E PREVIDÊNCIA S.A.',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('6572',
  'HDI SEGUROS S.A. " em aprovação "  (antiga HANNOVER INTERNATIONAL SEGUROS S.A.)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HDI SEGUROS S.A. " EM APROVAÇÃO "  (ANTIGA HANNOVER INTERNATIO...',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6661',
  'NOVA YORK CIA DE SEGUROS',
  '680',
  'NOVA YORK',
  'NOVA YORK | NOVA YORK CIA DE SEGUROS',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('8141',
  'CAIXAPREV VIDA E PREVIDÊNCIA S.A.',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXAPREV VIDA E PREVIDÊNCIA S.A.',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('8141',
  'CAIXAPREV VIDA E PREVIDÊNCIA S/A',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXAPREV VIDA E PREVIDÊNCIA S/A',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('8737',
  'AIG BRASIL COMPANHIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AIG BRASIL COMPANHIA DE SEGUROS',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('10031',
  'PECULIO ABRAHAM LINCOLN - AMAL',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PECULIO ABRAHAM LINCOLN - AMAL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10359',
  'CORRFA PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CORRFA PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'SOCIEDADE AUXILIADORA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOCIEDADE AUXILIADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10987',
  'SOCIEDADE CAXIENSE DE MÚTUO SOCORRO',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SOCIEDADE CAXIENSE DE MÚTUO SOCORRO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20281',
  'HORIZONTE CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HORIZONTE CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20451',
  'GLOBAL CAPITALIZACAO S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | GLOBAL CAPITALIZACAO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21318',
  'RURAL CAPITALIZAÇÃO S.A.',
  '710',
  'RURAL',
  'RURAL | RURAL CAPITALIZAÇÃO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'RURAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('25712',
  'REAL CAPITALIZAÇÃO S/A',
  '1200',
  'SANTANDER',
  'SANTANDER | REAL CAPITALIZAÇÃO S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('25941',
  'Capitaliza Empresa de Capitalização S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | CAPITALIZA EMPRESA DE CAPITALIZAÇÃO S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('4740',
  'KIRTON VIDA E PREVIDÊNCIA S.A. (Atual denominação da HSBC Vida e Previdência ...',
  '27',
  'BRADESCO',
  'BRADESCO | KIRTON VIDA E PREVIDÊNCIA S.A. (ATUAL DENOMINAÇÃO DA HSBC VIDA E P...',
  'BRADESCO SEGUROS',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'SIM'),
 ('1619',
  'DAYPREV VIDA E PREVIDÊNCIA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | DAYPREV VIDA E PREVIDÊNCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2062',
  'LUIZASEG SEGUROS S/A',
  '1203',
  'CARDIF',
  'CARDIF | LUIZASEG SEGUROS S/A',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('5011',
  'Chubb do Brasil Cia de Seguros',
  '221',
  'CHUBB',
  'CHUBB | CHUBB DO BRASIL CIA DE SEGUROS',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('5061',
  'SEGURADORA OCE¦NICA S/A',
  '451',
  'MERIDIONAL',
  'MERIDIONAL | SEGURADORA OCE¦NICA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MERIDIONAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10961',
  'PREVICORP Previdência Privada',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVICORP PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('42790',
  'SWISS REINSURANCE COMPANY LTD',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SWISS REINSURANCE COMPANY LTD',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('74951',
  'LENIX RE DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LENIX RE DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76767',
  'Cooper Gay do Brasil Corretora de Resseguros Limitada',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COOPER GAY DO BRASIL CORRETORA DE RESSEGUROS LIMITADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6548',
  'CARDIF DO BRASIL VIDA E PREVIDÊNCIA S/A',
  '1203',
  'CARDIF',
  'CARDIF | CARDIF DO BRASIL VIDA E PREVIDÊNCIA S/A',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('34665',
  'XL Resseguros Brasil S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XL RESSEGUROS BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3182',
  'ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A.',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | ITAÚ SEGUROS DE AUTO E RESIDÊNCIA S.A.',
  'ITAÚ UNIBANCO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'SIM'),
 ('79766',
  'Larim Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LARIM CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('40096',
  'Torus Specialty Insurance Company',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TORUS SPECIALTY INSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78743',
  'Aon Benfield Brasil Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AON BENFIELD BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5045',
  'COMPANHIA DE SEGUROS ALIANÇA DA BAHIA',
  '175',
  'ALIANÇA DA BAHIA',
  'ALIANÇA DA BAHIA | COMPANHIA DE SEGUROS ALIANÇA DA BAHIA',
  'OUTROS',
  'ALIANÇA DA BAHIA',
  'Aliança da Bahia',
  'Seguradora nacional independente',
  'Aliança da Bahia',
  'Alta',
  None,
  'SIM'),
 ('4812',
  'Euler Hermes Seguros de Credito à Exportação S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER HERMES SEGUROS DE CREDITO À EXPORTAÇÃO S.A.',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('22926',
  'BB Capitalização S.A. "em aprovação" (antiga Nossa Caixa Capitalização S.A.)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BB CAPITALIZAÇÃO S.A. "EM APROVAÇÃO" (ANTIGA NOSSA CAIXA CAPIT...',
  'CAIXA SEGURIDADE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Nossa Caixa legado',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Média',
  'Banco Nossa Caixa é legado histórico; não tratar como Caixa Econômica. Validar período societário.',
  'SIM'),
 ('2143',
  'Nationwide Seguradora S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NATIONWIDE SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5312',
  'BRADESCO AUTO/RE COMPANHIA DE SEGUROS',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO AUTO/RE COMPANHIA DE SEGUROS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('1627',
  'SAFRA SEGUROS GERAIS S.A.',
  '345',
  'SAFRA',
  'SAFRA | SAFRA SEGUROS GERAIS S.A.',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('38270',
  'SWISS RE BRASIL RESSEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SWISS RE BRASIL RESSEGUROS S/A',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('2879',
  'Comprev Seguradora S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPREV SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3417',
  'MG SEGUROS VIDA E PREVIDENCIA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MG SEGUROS VIDA E PREVIDENCIA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5886',
  'PORTO SEGURO COMPANHIA DE SEGUROS GERAIS',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO COMPANHIA DE SEGUROS GERAIS',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('6106',
  'KIRTON SEGUROS S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | KIRTON SEGUROS S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('41629',
  'SCOR Reinsurance Company',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SCOR REINSURANCE COMPANY',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('47082',
  'HDI GLOBAL NETWORK AG',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HDI GLOBAL NETWORK AG',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('3140',
  'PREVIMIL VIDA E PREVIDÊNCIA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVIMIL VIDA E PREVIDÊNCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5720',
  'Sompo Seguros S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOMPO SEGUROS S.A.',
  'OUTROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'SIM'),
 ('1805',
  'OXXY SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | OXXY SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1121',
  'YOUSE SEGURADORA S/A.',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | YOUSE SEGURADORA S/A.',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('42056',
  'ATRADIUS CREDITO Y CAUCION DE SEGUROS Y REASEGUROS SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ATRADIUS CREDITO Y CAUCION DE SEGUROS Y REASEGUROS SA',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('6696',
  'AXA XL SEGUROS S.A.',
  '1195',
  'AXA',
  'AXA | AXA XL SEGUROS S.A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('11126',
  'HOJE PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HOJE PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3531',
  'ZEMA SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZEMA SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3417',
  'BMG Seguradora S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BMG SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2534',
  'Efeito Seguradora Vida e Previdência S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EFEITO SEGURADORA VIDA E PREVIDÊNCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2071',
  'AVLA SEGUROS BRASIL S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AVLA SEGUROS BRASIL S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('47660',
  'Westport Insurance Corporation',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | WESTPORT INSURANCE CORPORATION',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6831',
  'SINAF SEGUROS S.A.',
  '1227',
  'SINAF',
  'SINAF | SINAF SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SINAF',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5185',
  'YELUM SEGUROS S.A.',
  '1230',
  'TALANX AG',
  'TALANX AG | YELUM SEGUROS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('45896',
  'GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO BRASIL LTDA',
  '43',
  'BAMERINDUS',
  'BAMERINDUS | GARD MARINE & ENERGY LIMITED - ESCRITÓRIO DE REPRESENTAÇÃO NO BR...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('74667',
  'THB RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | THB RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23540',
  'VIA CAPITALIZAÇÃO S/A (Em Aprovação)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VIA CAPITALIZAÇÃO S/A (EM APROVAÇÃO)',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2879',
  'Comprev Seguros e Previdência S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPREV SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('25429',
  'RG CAPITALIZAÇÃO S.A',
  '825',
  'ICATU',
  'ICATU | RG CAPITALIZAÇÃO S.A',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('77089',
  'BRIB CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BRIB CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6785',
  'Companhia de Seguros Aliança do Brasil',
  '78',
  'BRASIL',
  'BRASIL | COMPANHIA DE SEGUROS ALIANÇA DO BRASIL',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6017',
  'CENTAURO VIDA E PREVIDÊNCIA S. A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CENTAURO VIDA E PREVIDÊNCIA S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5720',
  'Yasuda Maritima Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | YASUDA MARITIMA SEGUROS S/A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('2771',
  'SUDAMERICA MICROSSEGURADORA DE DANOS E PESSOAS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUDAMERICA MICROSSEGURADORA DE DANOS E PESSOAS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5142',
  'Icatu Hartford Seguros S/A',
  '825',
  'ICATU',
  'ICATU | ICATU HARTFORD SEGUROS S/A',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('5177',
  'AGF BRASIL SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AGF BRASIL SEGUROS S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('5215',
  'ITAÚ VIDA E PREVIDÊNCIA S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ VIDA E PREVIDÊNCIA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5215',
  'ITAT PREVID-NCIA E SEGUROS S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAT PREVID-NCIA E SEGUROS S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5282',
  'PRUDENTIAL DO BRASIL SEGUROS DE VIDA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PRUDENTIAL DO BRASIL SEGUROS DE VIDA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5533',
  'ATLÂNTICA COMPANHIA DE SEGUROS "EM APROVAÇÃO" (ANTIGA FINASA SEGURADORA S/A)',
  '27',
  'BRADESCO',
  'BRADESCO | ATLÂNTICA COMPANHIA DE SEGUROS "EM APROVAÇÃO" (ANTIGA FINASA SEGUR...',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5649',
  'ROYAL & SUN ALLIANCE CIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ROYAL & SUN ALLIANCE CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5720',
  'Maritima Seguros S/A',
  '281',
  'MARÍTIMA',
  'MARÍTIMA | MARITIMA SEGUROS S/A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('5789',
  'amil seguradora s/a',
  '981',
  'AMIL',
  'AMIL | AMIL SEGURADORA S/A',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('6009',
  'BANERJ SEGUROS S/A',
  '35',
  'ITAÚ',
  'ITAÚ | BANERJ SEGUROS S/A',
  'BRADESCO SEGUROS',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'SIM'),
 ('6050',
  'MAXLIFE SEGURADORA DO BRASIL S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MAXLIFE SEGURADORA DO BRASIL S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6068',
  'Santos Seguradora S.A.',
  '1112',
  'SANTOS',
  'SANTOS | SANTOS SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6050',
  'MAXLIFE SEGURADORA DO BRASIL S/A',
  '1104',
  'MAXMED',
  'MAXMED | MAXLIFE SEGURADORA DO BRASIL S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MAXMED',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6122',
  'CIGNA SEGURADORA S.A.',
  '116',
  'CIGNA',
  'CIGNA | CIGNA SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CIGNA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6211',
  'Santa Catarina Seguros e Previdência S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SANTA CATARINA SEGUROS E PREVIDÊNCIA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6335',
  'SEGURADORA BRASILEIRA DE CRÉDITO A EXPOR',
  '27',
  'BRADESCO',
  'BRADESCO | SEGURADORA BRASILEIRA DE CRÉDITO A EXPOR',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVIDÊNCIA  S.A.',
  '78',
  'BRASIL',
  'BRASIL | AGF VIDA E PREVIDÊNCIA  S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('6459',
  'Cia de Seguros Tranquilidade - Brasil',
  '591',
  'INTER-ATLANTICO',
  'INTER-ATLANTICO | CIA DE SEGUROS TRANQUILIDADE - BRASIL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INTER-ATLANTICO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6491',
  'BBV PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BBV PREVIDÊNCIA E SEGURADORA BRASIL S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6491',
  'Alvorada Vida S.A. "em aprovação" (antig',
  '27',
  'BRADESCO',
  'BRADESCO | ALVORADA VIDA S.A. "EM APROVAÇÃO" (ANTIG',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6564',
  'Zurich Santander Brasil Seguros S/A (em aprovação) antiga Samtander Brasil Se...',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZURICH SANTANDER BRASIL SEGUROS S/A (EM APROVAÇÃO) ANTIGA SAM...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6611',
  'BEMGE SEGURADORA S/A',
  '35',
  'ITAÚ',
  'ITAÚ | BEMGE SEGURADORA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6653',
  'PANAMERICANA DE SEGUROS S-A',
  '1201',
  'BANCO PACTUAL',
  'BANCO PACTUAL | PANAMERICANA DE SEGUROS S-A',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('6831',
  'PESSOAL CIA. DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PESSOAL CIA. DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6866',
  'BRADESCO PREVIDÊNCIA E SEGUROS S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO PREVIDÊNCIA E SEGUROS S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6904',
  'SOMA SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOMA SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('9938',
  'Safra Seguros S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SAFRA SEGUROS S.A.',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('9938',
  'Safra Vida e Previdência S.A. (Safra Seguros S.A.) em Aprovação',
  '345',
  'SAFRA',
  'SAFRA | SAFRA VIDA E PREVIDÊNCIA S.A. (SAFRA SEGUROS S.A.) EM APROVAÇÃO',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('10413',
  'FAMILIA BA?DEIRANTE PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | FAMILIA BA?DEIRANTE PREVIDÊNCIA PRIVADA',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('10669',
  'RSPP PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | RSPP PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10669',
  'RS PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA RS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | RS PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA RS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDÊNCIA " em aprovação" ( antiga SOCIEDADE AUXILIADORA)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDÊNCIA " EM APROVAÇÃO" ( ANTIGA SOCIEDADE AUX...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11088',
  'UNIPREV UNIAO PREVIDENCIARIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UNIPREV UNIAO PREVIDENCIARIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('15580',
  'BMC PREVIDÊNCIA PRIVADA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BMC PREVIDÊNCIA PRIVADA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20451',
  'MEGACAP CAPITALIZAÇÃO S/A    "EM APROVAÇÃO"    (ANTIGA GLOBAL CAPITALIZACAO S/A)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MEGACAP CAPITALIZAÇÃO S/A    "EM APROVAÇÃO"    (ANTIGA GLOBAL...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('22764',
  'CREFICAP CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CREFICAP CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23345',
  'BCN CAPITALIZAÇÃO S.A',
  '27',
  'BRADESCO',
  'BRADESCO | BCN CAPITALIZAÇÃO S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('28134',
  'HSBC FINANCIAL CAPITALIZAÇÃO (BRASIL) S.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HSBC FINANCIAL CAPITALIZAÇÃO (BRASIL) S.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('70076',
  'MEXBRIT BRASIL CORRETORA DE RESSEGUROS LTDA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MEXBRIT BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5053',
  'Confiança Cia de Seguros',
  '540',
  'GBOEX-CONFIANCA',
  'GBOEX-CONFIANCA | CONFIANÇA CIA DE SEGUROS',
  'GBOEX-CONFIANÇA',
  'GBOEX / CONFIANÇA',
  'GBOEX / Confiança',
  'Seguradora/entidade tradicional',
  'GBOEX/Confiança',
  'Alta',
  None,
  'NÃO'),
 ('3727',
  'J.MALUCELLI VIDA E PREVIDENCIA S/A',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | J.MALUCELLI VIDA E PREVIDENCIA S/A',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('6181',
  'Brasilveículos Companhia de Seguros',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BRASILVEÍCULOS COMPANHIA DE SEGUROS',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('5495',
  'Zurich Minas Brasil Seguros S/A',
  '531',
  'ZURICH',
  'ZURICH | ZURICH MINAS BRASIL SEGUROS S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6912',
  'BMG SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BMG SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6947',
  'UNIMED SEGURADORA S/A',
  '795',
  'UNIMED',
  'UNIMED | UNIMED SEGURADORA S/A',
  'UNIMED',
  'UNIMED',
  'Unimed',
  'Saúde/operadora/cooperativa',
  'Unimed',
  'Alta',
  None,
  'NÃO'),
 ('10031',
  'VIVER PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIVER PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('49131',
  'ARCH REINSURANCE LTD.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARCH REINSURANCE LTD.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70858',
  'GUY CARPENTER & COMPANY CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GUY CARPENTER & COMPANY CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71439',
  'MILLER DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MILLER DO BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77747',
  'AD CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AD CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3387',
  'Angelus Seguros SA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ANGELUS SEGUROS SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5991',
  'Swiss Re Corporate Solutions Brasil Seguros S/A',
  '1229',
  'UBF',
  'UBF | SWISS RE CORPORATE SOLUTIONS BRASIL SEGUROS S/A',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('3727',
  'J.MALUCELLI SEGUROS S/A.',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | J.MALUCELLI SEGUROS S/A.',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('3794',
  'PREVIMAX PREVIDENCIA PRIVADA E SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVIMAX PREVIDENCIA PRIVADA E SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('48259',
  'TRANSATLANTIC REINSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TRANSATLANTIC REINSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('30074',
  'ALLIANZ GLOBAL CORPORATE & SPECIALTY RESSEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALLIANZ GLOBAL CORPORATE & SPECIALTY RESSEGUROS BRASIL S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('3298',
  'Mapfre Seguradora de Garantias e Crédito',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | MAPFRE SEGURADORA DE GARANTIAS E CRÉDITO',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('41629',
  'SCOR REINSURANCE COMPANY',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SCOR REINSURANCE COMPANY',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('47708',
  'Catlin Insurance Company (UK) Ltd.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CATLIN INSURANCE COMPANY (UK) LTD.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('72958',
  'ARX-RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARX-RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('73296',
  'BRISK RE CORRETORA DE RESSEGUROS LTDA (em APROVAÇÃO)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BRISK RE CORRETORA DE RESSEGUROS LTDA (EM APROVAÇÃO)',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74667',
  'Colemont Catalyst Re Corretora de Resseguro Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COLEMONT CATALYST RE CORRETORA DE RESSEGURO LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3794',
  'PREVIMAX PREVIDENCIA PRIVADA E SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PREVIMAX PREVIDENCIA PRIVADA E SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11029',
  'UNIAO DE PREVIDENCIA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIAO DE PREVIDENCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4561',
  'Nossa Caixa Seguros e Previdência S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | NOSSA CAIXA SEGUROS E PREVIDÊNCIA S.A.',
  'CAIXA SEGURIDADE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Nossa Caixa legado',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Média',
  'Banco Nossa Caixa é legado histórico; não tratar como Caixa Econômica. Validar período societário.',
  'SIM'),
 ('4928',
  'STARR INTERNATIONAL BRASIL SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | STARR INTERNATIONAL BRASIL SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70203',
  'CIRCLES GROUP BRASIL CORRETORA DE RESSEGURO S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CIRCLES GROUP BRASIL CORRETORA DE RESSEGURO S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5274',
  'Banestes Seguros S/A',
  '655',
  'BANESTES',
  'BANESTES | BANESTES SEGUROS S/A',
  'OUTROS',
  'BANESTES',
  'Banestes Seguros',
  'Bancassurance / banco regional',
  'Banestes',
  'Alta',
  None,
  'SIM'),
 ('21318',
  'KOVR CAPITALIZAÇÃO S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | KOVR CAPITALIZAÇÃO S.A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('6653',
  'Too Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TOO SEGUROS S.A.',
  'OUTROS',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'SIM'),
 ('20826',
  'MONGERAL AEGON CAPITALIZAÇÃO S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MONGERAL AEGON CAPITALIZAÇÃO S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('49000',
  'GENERAL INSURANCE CORPORATION OF INDIA ESCRITORIO DE REPRESENTACAO NO BRASIL ...',
  '78',
  'BRASIL',
  'BRASIL | GENERAL INSURANCE CORPORATION OF INDIA ESCRITORIO DE REPRESENTACAO N...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BRASIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71269',
  'BKS RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BKS RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4359',
  'Euler Hermes Seguros S.A.',
  '1228',
  'EULER HERMES',
  'EULER HERMES | EULER HERMES SEGUROS S.A.',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('3727',
  'TRAVELERS SEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TRAVELERS SEGUROS BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6696',
  'AXA CORPORATE SOLUTIONS SEGUROS S.A.',
  '1195',
  'AXA',
  'AXA | AXA CORPORATE SOLUTIONS SEGUROS S.A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('6785',
  'Companhia de Seguros Aliança do Brasil',
  '1233',
  'BBMAPFRE',
  'BBMAPFRE | COMPANHIA DE SEGUROS ALIANÇA DO BRASIL',
  'MAPFRE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('4855',
  'Now Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NOW SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1775',
  'ITAU SEGUROS SOLUÇÕES CORPORATIVAS S/A',
  '35',
  'ITAÚ',
  'ITAÚ | ITAU SEGUROS SOLUÇÕES CORPORATIVAS S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('45004',
  'INTACT INSURANCE UK LIMITED',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | INTACT INSURANCE UK LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5754',
  'NOBRE SEGURADORA DO BRASIL S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NOBRE SEGURADORA DO BRASIL S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4359',
  'Euler Hermes Seg.Créd.S/A "em aprov"(ant',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER HERMES SEG.CRÉD.S/A "EM APROV"(ANT',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('3727',
  'TRAVELERS SEGUROS BRASILS.A.',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | TRAVELERS SEGUROS BRASILS.A.',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('3298',
  'Mapfre Vera Cruz Previdência S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ PREVIDÊNCIA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('3816',
  'Real Tokio Marine Vida e Previdência S.A.',
  '86',
  'REAL',
  'REAL | REAL TOKIO MARINE VIDA E PREVIDÊNCIA S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5312',
  'BRADESCO AUTO/RE COMPANHIA DE SEGUROS',
  '43',
  'BAMERINDUS',
  'BAMERINDUS | BRADESCO AUTO/RE COMPANHIA DE SEGUROS',
  'BRADESCO SEGUROS',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'SIM'),
 ('21318',
  'INVESTPREV CAPITALIZAÇÃO S/A',
  '710',
  'RURAL',
  'RURAL | INVESTPREV CAPITALIZAÇÃO S/A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('5525',
  'INTERBRAZIL SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | INTERBRAZIL SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5525',
  'INTERBRAZIL SEGURADORA S/A',
  '1226',
  'INTERBRAZIL',
  'INTERBRAZIL | INTERBRAZIL SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INTERBRAZIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5941',
  'RELIANCE NATIONAL BRASIL SEGUROS S.A.',
  '1058',
  'BMC',
  'BMC | RELIANCE NATIONAL BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BMC',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5983',
  'UNIBANCO SEGUROS',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO SEGUROS',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6033',
  'PORTO SEGURO VIDA E PREVID-NCIA S/A.',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO VIDA E PREVID-NCIA S/A.',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('6092',
  'NOVO HAMBURGO COMPANHIA DE SEGUROS GERAI',
  '27',
  'BRADESCO',
  'BRADESCO | NOVO HAMBURGO COMPANHIA DE SEGUROS GERAI',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6190',
  'TOKIO MARINE SEGURADORA S.A. "EM APROVAÇÃO" (NOVA DENOMINAÇÃO SOCIAL DA REAL ...',
  '1222',
  'TOKIO MARINE',
  'TOKIO MARINE | TOKIO MARINE SEGURADORA S.A. "EM APROVAÇÃO" (NOVA DENOMINAÇÃO ...',
  'SANTANDER',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'SIM'),
 ('6211',
  'Santa Catarina Seguros e Previdência S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SANTA CATARINA SEGUROS E PREVIDÊNCIA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6220',
  'SUL AMÉRICA CIA NACIONAL DE SEGUROS GERA',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA CIA NACIONAL DE SEGUROS GERA',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6271',
  'SUDAMERIS VIDA E PREVIDENCIA S/A "em aprovação"(SUDAMERIS GENERALI CIA.NAC.SE...',
  '86',
  'REAL',
  'REAL | SUDAMERIS VIDA E PREVIDENCIA S/A "EM APROVAÇÃO"(SUDAMERIS GENERALI CIA...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6335',
  'SEGURADORA BRASILEIRA DE CRÉDITO A EXPOR',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA BRASILEIRA DE CRÉDITO A EXPOR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6378',
  'NATIONWIDE MARÍTIMA VIDA E PREVIDÊNCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NATIONWIDE MARÍTIMA VIDA E PREVIDÊNCIA',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6378',
  'Dinamica Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | DINAMICA SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVIDÊNCIA  S.A.',
  '1190',
  'AGF BRASIL',
  'AGF BRASIL | AGF VIDA E PREVIDÊNCIA  S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('6491',
  'BBV PREVID-NCIA E SEGURADORA BRASIL S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BBV PREVID-NCIA E SEGURADORA BRASIL S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6645',
  'CIA REAL BRASILEIRA DE SEGUROS',
  '86',
  'REAL',
  'REAL | CIA REAL BRASILEIRA DE SEGUROS',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6688',
  'COSESP - CIA DE SEGUROS DO ESTADO DE S¦O',
  '159',
  'COSESP',
  'COSESP | COSESP - CIA DE SEGUROS DO ESTADO DE S¦O',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'COSESP',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6858',
  'Seguradora Roma S/A',
  '752',
  'ROMA',
  'ROMA | SEGURADORA ROMA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'ROMA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6891',
  'SAOEX S.A. SEGURADORA E PREVIDÊNCIA PRIV',
  '728',
  'SAOEX',
  'SAOEX | SAOEX S.A. SEGURADORA E PREVIDÊNCIA PRIV',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SAOEX',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('8737',
  'AMERICAN HOME DO BRASIL S/A',
  '108',
  'INTERAMERICANA',
  'INTERAMERICANA | AMERICAN HOME DO BRASIL S/A',
  'OUTROS',
  'AIG',
  'AIG / American Home / Interamericana',
  'Seguradora global especializada',
  'AIG e legados',
  'Alta',
  None,
  'SIM'),
 ('10138',
  'BAMÉRCIO S/A PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BAMÉRCIO S/A PREVIDÊNCIA PRIVADA',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('10235',
  'CAPEMI-CX.PEC.PENS.E MONTEP.-BENEFICENTE',
  '1188',
  'CAPEMI',
  'CAPEMI | CAPEMI-CX.PEC.PENS.E MONTEP.-BENEFICENTE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CAPEMI',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10405',
  'Empresarial de Previdencia Privada',
  '1040',
  'EMPRESARIAL',
  'EMPRESARIAL | EMPRESARIAL DE PREVIDENCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'EMPRESARIAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'SOCIEDADE AUXILIADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SOCIEDADE AUXILIADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDENCIA  "em aprovação " (antiga SOCIEDADE AUXILIADORA)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDENCIA  "EM APROVAÇÃO " (ANTIGA SOCIEDADE AUX...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11304',
  'Luterprev - Entidade Luterana de Previde',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LUTERPREV - ENTIDADE LUTERANA DE PREVIDE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20273',
  'Valor Capitalização S.A.',
  '1112',
  'SANTOS',
  'SANTOS | VALOR CAPITALIZAÇÃO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('22896',
  'Itaú Capitalização S/A',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ CAPITALIZAÇÃO S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('73296',
  'SWIINGLEHURST BRASIL RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SWIINGLEHURST BRASIL RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('75434',
  'WILLIS CORRETORES DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | WILLIS CORRETORES DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78743',
  'Aon Re Brasil Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AON RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5070',
  'BOZANO, SIMONSEN SEGURADORA S/A',
  '451',
  'MERIDIONAL',
  'MERIDIONAL | BOZANO, SIMONSEN SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MERIDIONAL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6831',
  'SINAF PREVIDENCIAL CIA DE SEGUROS',
  '1227',
  'SINAF',
  'SINAF | SINAF PREVIDENCIAL CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SINAF',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10413',
  'FAMILIA BANDEIRANTE PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | FAMILIA BANDEIRANTE PREVIDÊNCIA PRIVADA',
  'AMIL',
  'AMIL / UNITEDHEALTH',
  'Amil / UnitedHealth',
  'Saúde/operadora',
  'Amil/United/Blue Life',
  'Alta',
  None,
  'NÃO'),
 ('10448',
  'GBOEX - GREMIO BENEFICENTE',
  '540',
  'GBOEX-CONFIANCA',
  'GBOEX-CONFIANCA | GBOEX - GREMIO BENEFICENTE',
  'GBOEX-CONFIANÇA',
  'GBOEX / CONFIANÇA',
  'GBOEX / Confiança',
  'Seguradora/entidade tradicional',
  'GBOEX/Confiança',
  'Alta',
  None,
  'NÃO'),
 ('10987',
  'SOCIEDADE CAXIENSE DE MÚTUO SOCORRO',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOCIEDADE CAXIENSE DE MÚTUO SOCORRO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('44814',
  'PARTNER REINSURANCE EUROPE SE',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PARTNER REINSURANCE EUROPE SE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6530',
  'VIRGINIA SURETY COMPANHIA DE SEGUROS DO BRASIL',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIRGINIA SURETY COMPANHIA DE SEGUROS DO BRASIL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2437',
  'BTG Pactual Seguradora S.A.',
  '1201',
  'BANCO PACTUAL',
  'BANCO PACTUAL | BTG PACTUAL SEGURADORA S.A.',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('1490',
  'Essor Seguros S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ESSOR SEGUROS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3271',
  'SEGURADORA LÍDER DOS CONSÓRCIOS DO SEGURO DPVAT S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA LÍDER DOS CONSÓRCIOS DO SEGURO DPVAT S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77054',
  'HOWDEN CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HOWDEN CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5193',
  'CIA SEGUROS PREVIDENCIA DO SUL',
  '990',
  'APLUB',
  'APLUB | CIA SEGUROS PREVIDENCIA DO SUL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'APLUB',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3182',
  'ITAU SEGUROS DE AUTO E RESIDÊNCIA',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | ITAU SEGUROS DE AUTO E RESIDÊNCIA',
  'ITAÚ UNIBANCO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'SIM'),
 ('28258',
  'MAPFRE Capitalização S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE CAPITALIZAÇÃO S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('72877',
  'CORMATT CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CORMATT CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78271',
  'JLT RE BRASIL ADMINISTRAÇÃO E CORRETAGEM DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | JLT RE BRASIL ADMINISTRAÇÃO E CORRETAGEM DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23566',
  'BRADESCO CAPITALIZAÇÃO S.A',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO CAPITALIZAÇÃO S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('26158',
  'HSBC EMPRESA DE CAPITALIZAÇÃO S.A.',
  '1204',
  'HSBC',
  'HSBC | HSBC EMPRESA DE CAPITALIZAÇÃO S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('4561',
  'Mapfre Nossa Caixa Vida e Previdência S.A',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE NOSSA CAIXA VIDA E PREVIDÊNCIA S.A',
  'CAIXA SEGURIDADE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Média',
  'Contém Mapfre Nossa Caixa; tratar como legado Mapfre/BB e validar período.',
  'SIM'),
 ('2275',
  'ALM SEGURADORA S.A - MICROSSEGURADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALM SEGURADORA S.A - MICROSSEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('47406',
  'FINANCIAL INSURANCE COMPANY LIMITED',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FINANCIAL INSURANCE COMPANY LIMITED',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('40215',
  'MAPFRE RE COMPAÑÍA DE REASEGUROS S.A',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE RE COMPAÑÍA DE REASEGUROS S.A',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('30201',
  'TERRA BRASIS RESSEGUROS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TERRA BRASIS RESSEGUROS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('40096',
  'STARSTONE SPECIALTY INSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | STARSTONE SPECIALTY INSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77402',
  'TRUSTER BRASIL CORRETAGEM DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TRUSTER BRASIL CORRETAGEM DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('38253',
  'CHUBB RESSEGURADORA BRASIL S.A',
  '221',
  'CHUBB',
  'CHUBB | CHUBB RESSEGURADORA BRASIL S.A',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('4481',
  'DASSEG SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | DASSEG SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('28932',
  "RIO'S CAPITALIZAÇÃO S.A.",
  '19',
  'SUL AMERICA',
  "SUL AMERICA | RIO'S CAPITALIZAÇÃO S.A.",
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6157',
  'ZURICH BRASIL VIDA E PREVIDÊNCIA S.A',
  '531',
  'ZURICH',
  'ZURICH | ZURICH BRASIL VIDA E PREVIDÊNCIA S.A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('75825',
  'QUALISEG BRASIL RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | QUALISEG BRASIL RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1091',
  'SUIÇA SEGURADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUIÇA SEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74381',
  'ONEGLOBAL BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ONEGLOBAL BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1554',
  'EQUATORIAL MICROSSEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EQUATORIAL MICROSSEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6912',
  'ITAU BMG SEGURADORA SA',
  '35',
  'ITAÚ',
  'ITAÚ | ITAU BMG SEGURADORA SA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('3727',
  'TRAVELERS SEGUROS BRASIL S/A (em aprovação da SUSEP - antiga J.MALUCELLI SEGU...',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | TRAVELERS SEGUROS BRASIL S/A (EM APROVAÇÃO DA SUSEP - ANTIGA ...',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('3816',
  'Real Tokio Marine Vida e Previdência S.A.',
  '1200',
  'SANTANDER',
  'SANTANDER | REAL TOKIO MARINE VIDA E PREVIDÊNCIA S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('2143',
  'Nationwide Seguradora S.A.',
  '1213',
  'NATIONWIDE',
  'NATIONWIDE | NATIONWIDE SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NATIONWIDE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2763',
  'CESCEBRASIL SEGUROS DE CRÉDITO S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CESCEBRASIL SEGUROS DE CRÉDITO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('19',
  'SUL AMERICA',
  None,
  None,
  '| SUL AMERICA',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5096',
  'UNIBANCO VIDA E PREVIDÊNCIA S/A',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO VIDA E PREVIDÊNCIA S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5177',
  'AGF BRASIL SEGUROS S.A.',
  '1190',
  'AGF BRASIL',
  'AGF BRASIL | AGF BRASIL SEGUROS S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('5177',
  'ALLIANZ SEGUROS S.A."em aprovação" (antiga AGF BRASIL SEGUROS S.A.)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ALLIANZ SEGUROS S.A."EM APROVAÇÃO" (ANTIGA AGF BRASIL SEGUROS ...',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('5185',
  'LIBERTY SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | LIBERTY SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5533',
  'FINASA SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | FINASA SEGURADORA S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5843',
  'Indiana Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | INDIANA SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('5941',
  'RELIANCE NATIONAL BRASIL SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | RELIANCE NATIONAL BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5983',
  'UNIBANCO AIG SEGUROS S/A',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO AIG SEGUROS S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5983',
  'UNIBANCO SEGUROS S/A "Em Aprovação" (Antiga Unibanco AIG Seguros S/A)',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO SEGUROS S/A "EM APROVAÇÃO" (ANTIGA UNIBANCO AIG S...',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5991',
  'SEGURADORA BRASILEIRA DE FIANÇAS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEGURADORA BRASILEIRA DE FIANÇAS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5991',
  'Swiss Re Corporate Solutions Brasil Seguros S/A "em aprovação" (antiga UBF Se...',
  '1229',
  'UBF',
  'UBF | SWISS RE CORPORATE SOLUTIONS BRASIL SEGUROS S/A "EM APROVAÇÃO" (ANTIGA ...',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6050',
  'MAXMED SEGURADORA S/A',
  '1023',
  'NOBRE',
  'NOBRE | MAXMED SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NOBRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6149',
  'TREVO S/A SEGUROS E PREVID-NCIA PRIVADA',
  '426',
  'TREVO',
  'TREVO | TREVO S/A SEGUROS E PREVID-NCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'TREVO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6157',
  'MINAS BRASIL SEG.VIDA E PREVIDÊNCIA S.A',
  '531',
  'ZURICH',
  'ZURICH | MINAS BRASIL SEG.VIDA E PREVIDÊNCIA S.A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6157',
  'MINAS BRASIL SEG.VIDA E PREVIDÊNCIA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MINAS BRASIL SEG.VIDA E PREVIDÊNCIA S.A',
  'OUTROS',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'SIM'),
 ('6271',
  'SUDAMERIS GENERALI CIA.NAC.SEGURO E PREV',
  '213',
  'GENERALLI',
  'GENERALLI | SUDAMERIS GENERALI CIA.NAC.SEGURO E PREV',
  'GENERALI',
  'GENERALI',
  'Generali',
  'Seguradora global multilinha',
  'Generali',
  'Alta',
  None,
  'NÃO'),
 ('6351',
  'METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA',
  '1215',
  'METROPOLITAN LIFE',
  'METROPOLITAN LIFE | METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA',
  'METLIFE',
  'METLIFE',
  'MetLife / Metropolitan',
  'Vida e benefícios',
  'MetLife/Metropolitan',
  'Alta',
  None,
  'NÃO'),
 ('6432',
  'NATIONALE NEDERLANDEN LEVENSVERZEKERING',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NATIONALE NEDERLANDEN LEVENSVERZEKERING',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6432',
  'NATIONALE NEDERLANDEN LEVENSVERZEKERING',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NATIONALE NEDERLANDEN LEVENSVERZEKERING',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6513',
  'ACE SEGURADORA S.ª',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ACE SEGURADORA S.ª',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('6793',
  'Gente Seguradora S.A.',
  '558',
  'GENTE',
  'GENTE | GENTE SEGURADORA S.A.',
  'OUTROS',
  'GENTE SEGURADORA',
  'Gente Seguradora',
  'Seguradora nacional independente',
  'Gente Seguradora',
  'Alta',
  None,
  'SIM'),
 ('6831',
  'SINAF PREVIDENCIAL CIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SINAF PREVIDENCIAL CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6912',
  'CONAPP CIA NACIONAL DE SEGUROS',
  '1188',
  'CAPEMI',
  'CAPEMI | CONAPP CIA NACIONAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CAPEMI',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6955',
  'CIA DE SEGUROS GRALHA AZUL',
  '817',
  'GRALHA AZUL',
  'GRALHA AZUL | CIA DE SEGUROS GRALHA AZUL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'GRALHA AZUL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6998',
  'AUREA SEGUROS S/A',
  '809',
  'ÁUREA',
  'ÁUREA | AUREA SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'ÁUREA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10120',
  'EQUATORIAL PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EQUATORIAL PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('18091',
  'BOSTON PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BOSTON PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('18091',
  'BOSTON PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BOSTON PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20451',
  'GLOBAL CAPITALIZACAO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GLOBAL CAPITALIZACAO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21482',
  'UNIBANCO CIA DE CAPITALIZAÇÃO',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO CIA DE CAPITALIZAÇÃO',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('25585',
  'FEDERAL CAPITALIZAÇÃO',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | FEDERAL CAPITALIZAÇÃO',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('74811',
  'UIB RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UIB RE BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3093',
  'CONSORCIOS DE OP. DO SEGURO OBRIG. DE DANOS PESSOAIS CAUSADOS POR VEÍCUL AUTO...',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CONSORCIOS DE OP. DO SEGURO OBRIG. DE DANOS PESSOAIS CAUSADOS...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5045',
  'COMPANHIA DE SEGUROS ALIANÇA DA BAHIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPANHIA DE SEGUROS ALIANÇA DA BAHIA',
  'OUTROS',
  'ALIANÇA DA BAHIA',
  'Aliança da Bahia',
  'Seguradora nacional independente',
  'Aliança da Bahia',
  'Alta',
  None,
  'SIM'),
 ('44920',
  'Everest Reinsurance Company',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EVEREST REINSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('48275',
  'ZURICH INSURANCE COMPANY LTD.',
  '531',
  'ZURICH',
  'ZURICH | ZURICH INSURANCE COMPANY LTD.',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6718',
  'Mapfre Seguradora de Crédito à Exportação S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE SEGURADORA DE CRÉDITO À EXPORTAÇÃO S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6238',
  'MAPFRE SEGUROS GERAIS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE SEGUROS GERAIS S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('8141',
  'CAIXA VIDA E PREVIDÊNCIA S/A',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXA VIDA E PREVIDÊNCIA S/A',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('10103',
  'ARCESP PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ARCESP PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11096',
  'Upofa - União Previdencial',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UPOFA - UNIÃO PREVIDENCIAL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('44661',
  'SCOR GLOBAL LIFE AMERICAS REINSURANCE COMPANY',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SCOR GLOBAL LIFE AMERICAS REINSURANCE COMPANY',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('6564',
  'Zurich Santander Brasil Seguros S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZURICH SANTANDER BRASIL SEGUROS S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('31623',
  'IRB BRASIL RESSEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | IRB BRASIL RESSEGUROS S/A',
  'OUTROS',
  'IRB RE',
  'IRB Brasil RE',
  'Resseguro',
  'IRB',
  'Alta',
  None,
  'SIM'),
 ('2879',
  'COMPREV SEGUROS E PREVIDENCIA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMPREV SEGUROS E PREVIDENCIA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4251',
  'CAPEMISA SEGURADORA DE VIDA E PREVIDÊNCIA S/A',
  '1188',
  'CAPEMI',
  'CAPEMI | CAPEMISA SEGURADORA DE VIDA E PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CAPEMI',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70076',
  'MEXBRIT BRASIL CORRETORA DE RESSEGUROS LTDA.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | MEXBRIT BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74926',
  'ASSURÊ INTERNACIONAL CORRETAGEM DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ASSURÊ INTERNACIONAL CORRETAGEM DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78255',
  'PECUS CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PECUS CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5321',
  'ITAU SEGUROS S/A',
  '35',
  'ITAÚ',
  'ITAÚ | ITAU SEGUROS S/A',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('4952',
  'SUHAI SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUHAI SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1937',
  'COMPREV VIDA E PREVIDENCIA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COMPREV VIDA E PREVIDENCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2267',
  'VOCÊ SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VOCÊ SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1279',
  'SOMBRERO SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SOMBRERO SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2968',
  'GAZIN SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GAZIN SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2321',
  'CICLLOS SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CICLLOS SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('37753',
  'Sompo Resseguradora S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SOMPO RESSEGURADORA S.A',
  'OUTROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'SIM'),
 ('6173',
  'KOVR PREVIDÊNCIA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | KOVR PREVIDÊNCIA S.A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('3654',
  'SICOOB SEGURADORA DE VIDA E PREVIDENCIA S. A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SICOOB SEGURADORA DE VIDA E PREVIDENCIA S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2020',
  'ATRADIUS CRÉDITO Y CAUCIÓN SEGURADORA S.A.',
  '1231',
  'CRÉDITO Y CAUCIÓN',
  'CRÉDITO Y CAUCIÓN | ATRADIUS CRÉDITO Y CAUCIÓN SEGURADORA S.A.',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('4031',
  'XP VIDA E PREVIDÊNCIA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XP VIDA E PREVIDÊNCIA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5843',
  'Indiana Seguros S/A',
  '1230',
  'TALANX AG',
  'TALANX AG | INDIANA SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('6513',
  'CHUBB SEGUROS BRASIL S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CHUBB SEGUROS BRASIL S.A.',
  'CHUBB',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'BB Seguridade',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('1261',
  'FACTA SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FACTA SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78999',
  'ACRISURE BRASIL CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ACRISURE BRASIL CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76236',
  'BMS BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BMS BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1261',
  'FACTA SEGURADORA S/A - MICROSSEGURADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FACTA SEGURADORA S/A - MICROSSEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('48178',
  'RGA GLOBAL REINSURANCE COMPANY, LTD.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | RGA GLOBAL REINSURANCE COMPANY, LTD.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5436',
  'Junto Seguros S.A.',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | JUNTO SEGUROS S.A.',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('6084',
  'MBM SEGURADORA S/A',
  '1121',
  'MBM',
  'MBM | MBM SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'MBM',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74381',
  'Oneglobal Brasil Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ONEGLOBAL BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1571',
  'HDI GLOBAL SEGUROS S.A',
  '1230',
  'TALANX AG',
  'TALANX AG | HDI GLOBAL SEGUROS S.A',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('44661',
  'SCOR GLOBAL LIFE AMERICAS REINSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SCOR GLOBAL LIFE AMERICAS REINSURANCE COMPANY',
  'OUTROS',
  'SCOR',
  'SCOR Re',
  'Resseguro',
  'SCOR',
  'Alta',
  None,
  'SIM'),
 ('2275',
  'SEGURADORA ALM S/A (em aprovação)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SEGURADORA ALM S/A (EM APROVAÇÃO)',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2925',
  'VOLTS SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VOLTS SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5819',
  'ALLSEG SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ALLSEG SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1091',
  'Suíça Seguradora',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUÍÇA SEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1091',
  'ARKA SEGURADORA S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARKA SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77054',
  'HOWDEN CORRETORA DE RESSEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HOWDEN CORRETORA DE RESSEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4618',
  'ITAÚ XL SEGUROS CORPORATIVOS S.A. (em alteração para ITAÚ UNIBANCO SEGUROS CO...',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ XL SEGUROS CORPORATIVOS S.A. (EM ALTERAÇÃO PARA ITAÚ UNIBANCO SEG...',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('1929',
  'HDI SEGUROS DE AUTOMÓVEIS E BENS S.A. "EM APROVAÇÃO" (ANTIGA HSBC SEGUROS DE ...',
  '1210',
  'HANNOVER',
  'HANNOVER | HDI SEGUROS DE AUTOMÓVEIS E BENS S.A. "EM APROVAÇÃO" (ANTIGA HSBC ...',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5151',
  'AMÉRICA LATINA COMPANHIA DE SEGUROS',
  '311',
  'AMÉRICA LATINA',
  'AMÉRICA LATINA | AMÉRICA LATINA COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'AMÉRICA LATINA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5215',
  'ITAÚ PREVIDÊNCIA E SEGUROS S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ PREVIDÊNCIA E SEGUROS S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5215',
  'ITAÚ VIDA E PREVIDÊNCIA S.A. antiga (ITA',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ VIDA E PREVIDÊNCIA S.A. ANTIGA (ITA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5291',
  'São Paulo Cia Nacional de Seguros Gerais',
  '124',
  'SÃO PAULO',
  'SÃO PAULO | SÃO PAULO CIA NACIONAL DE SEGUROS GERAIS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SÃO PAULO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5312',
  'UNIÃO NOVO HAMBURGO SEGUROS S/A',
  '27',
  'BRADESCO',
  'BRADESCO | UNIÃO NOVO HAMBURGO SEGUROS S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5401',
  'PQ SEGUROS S.A. "em aprovação "(antiga B',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PQ SEGUROS S.A. "EM APROVAÇÃO "(ANTIGA B',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5541',
  'CCF Brasil Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CCF BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5665',
  'MAPFRE VIDA S.A. - (EM APROVAÇÃO)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE VIDA S.A. - (EM APROVAÇÃO)',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5711',
  'BRADESCO-ATLANTICA CIA DE SEGUROS',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO-ATLANTICA CIA DE SEGUROS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5754',
  'NOBRE SEGURADORA DO BRASIL S/A',
  '1023',
  'NOBRE',
  'NOBRE | NOBRE SEGURADORA DO BRASIL S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NOBRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5801',
  'TREVO SEGURADORA S/A',
  '426',
  'TREVO',
  'TREVO | TREVO SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'TREVO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6041',
  'PARANA CIA DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PARANA CIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6122',
  'FATOR SEGURADORA S/A "em aprovação" (Antiga CIGNA Seguradora S/A)',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FATOR SEGURADORA S/A "EM APROVAÇÃO" (ANTIGA CIGNA SEGURADORA ...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6203',
  'ZURICH BRASIL SEGUROS S/A',
  '531',
  'ZURICH',
  'ZURICH | ZURICH BRASIL SEGUROS S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6220',
  'SUL AMÉRICA CIA NACIONAL DE SEGS. E PREV',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA CIA NACIONAL DE SEGS. E PREV',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6378',
  'NATIONWIDE MARÍTIMA VIDA E PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NATIONWIDE MARÍTIMA VIDA E PREVIDÊNCIA',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6378',
  'VIDA SEGURADORA S.A. "Em aprovação" (Antiga Nationwide Marítima Vida e Previd...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VIDA SEGURADORA S.A. "EM APROVAÇÃO" (ANTIGA NATIONWIDE MARÍTIM...',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6408',
  'CIGNA COMPANHIA DE SEGUROS S/A',
  '116',
  'CIGNA',
  'CIGNA | CIGNA COMPANHIA DE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CIGNA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6564',
  'Santander Banespa Seguros S/A',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER BANESPA SEGUROS S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6751',
  'ROYAL & SUNALLIANCE SEGUROS (BRASIL) S.A',
  '515',
  'SUN ALLIANCE',
  'SUN ALLIANCE | ROYAL & SUNALLIANCE SEGUROS (BRASIL) S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SUN ALLIANCE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6777',
  'APS SEGURADORA S.A. (em aprovação) CAOA',
  '1221',
  'SULINA',
  'SULINA | APS SEGURADORA S.A. (EM APROVAÇÃO) CAOA',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6840',
  'Unibanco AIG SA Seguros e Previdência',
  '701',
  'PREVER',
  'PREVER | UNIBANCO AIG SA SEGUROS E PREVIDÊNCIA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6858',
  'MAPFRE AFFINITY SEGURADORA S.A. "EM APROVAÇÃO"',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE AFFINITY SEGURADORA S.A. "EM APROVAÇÃO"',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6866',
  'BRADESCO PREVID-NCIA E SEGUROS S.A.',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO PREVID-NCIA E SEGUROS S.A.',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6939',
  'Seguradora América do Sul S.A.',
  '256',
  'AMÉRICA DO SUL',
  'AMÉRICA DO SUL | SEGURADORA AMÉRICA DO SUL S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6955',
  'CIA DE SEGUROS GRALHA AZUL',
  '35',
  'ITAÚ',
  'ITAÚ | CIA DE SEGUROS GRALHA AZUL',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('10120',
  'EQUATORIAL PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EQUATORIAL PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10596',
  'PREVBRAS SOC NAC DE PREV PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PREVBRAS SOC NAC DE PREV PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10847',
  'PREVIMIL PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVIMIL PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10847',
  'PREVIMIL PREVIDÊNCIA COMPLEMENTAR S.A. "EM APROVAÇÃO" (ANTIGA PREVIMIL PREVID...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVIMIL PREVIDÊNCIA COMPLEMENTAR S.A. "EM APROVAÇÃO" (ANTIGA ...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10898',
  'RECÍPROCA ASSISTÊNCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | RECÍPROCA ASSISTÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('14257',
  'BP PREVIDÊNCIA PRIVADA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BP PREVIDÊNCIA PRIVADA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20451',
  'Arca Capitalização S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARCA CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('41491',
  'HANNOVER RUCKVERSINCHERUNG AG',
  '1210',
  'HANNOVER',
  'HANNOVER | HANNOVER RUCKVERSINCHERUNG AG',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('70211',
  'PWS CORRETORA DE RESSEGUROS LTDA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PWS CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('71412',
  'ADAMS & PORTER CORRETORA DE RESSEGUROS LTDA.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ADAMS & PORTER CORRETORA DE RESSEGUROS LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('40851',
  'CHUBB TEMPEST REINSURANCE LTD. - Escritório Representação no Brasil Ltda.',
  '221',
  'CHUBB',
  'CHUBB | CHUBB TEMPEST REINSURANCE LTD. - ESCRITÓRIO REPRESENTAÇÃO NO BRASIL L...',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('38873',
  'JUNTO RESSEGUROS S.A. "em aprovação" (antiga J.MALUCELLI RESSEGURADORA S/A)',
  '965',
  'J. MALUCELLI',
  'J. MALUCELLI | JUNTO RESSEGUROS S.A. "EM APROVAÇÃO" (ANTIGA J.MALUCELLI RESSE...',
  'OUTROS',
  'JUNTO / J. MALUCELLI',
  'Junto Seguros / J. Malucelli',
  'Garantia / seguradora especializada',
  'Junto/J. Malucelli',
  'Alta',
  None,
  'SIM'),
 ('2143',
  'ASSURANT SEGURADORA SA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ASSURANT SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5070',
  'Santander Seguros S/A (em aprov. Bozano)',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER SEGUROS S/A (EM APROV. BOZANO)',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('4871',
  'VOTORANTIM SEGUROS E PREVIDENCIA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VOTORANTIM SEGUROS E PREVIDENCIA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4871',
  'VOTORANTIM SEGUROS E PREVIDÊNCIA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | VOTORANTIM SEGUROS E PREVIDÊNCIA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5070',
  'Zurich Santander Brasil Seguros e Previdencia S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZURICH SANTANDER BRASIL SEGUROS E PREVIDENCIA S/A',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'COMPANHIA BRASILEIRA DE SEGUROS E PREVIDÊNCIA',
  '825',
  'ICATU',
  'ICATU | COMPANHIA BRASILEIRA DE SEGUROS E PREVIDÊNCIA',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('3328',
  'Coface do Brasil Seguros de Crédito S/A (atual denominação da Coface do Brasi...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COFACE DO BRASIL SEGUROS DE CRÉDITO S/A (ATUAL DENOMINAÇÃO DA ...',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('22926',
  'BB Capitalização S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BB CAPITALIZAÇÃO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21661',
  'Cia Itaú de Capitalização',
  '35',
  'ITAÚ',
  'ITAÚ | CIA ITAÚ DE CAPITALIZAÇÃO',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('2950',
  'SANCOR SEGUROS DO BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SANCOR SEGUROS DO BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4952',
  'SUHAI SEGUROS SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUHAI SEGUROS SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6157',
  'ZURICH VIDA E PREVIDÊNCIA S.A',
  '531',
  'ZURICH',
  'ZURICH | ZURICH VIDA E PREVIDÊNCIA S.A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('11070',
  'União Previdenciária Cometa do Brasil',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIÃO PREVIDENCIÁRIA COMETA DO BRASIL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('79553',
  'Promass Corretora de Resseguros Ltda',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PROMASS CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2721',
  'CRÉDITO Y CAUCIÓN SEGURADORA DE CRÉDITO À EXPORTAÇÃO S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CRÉDITO Y CAUCIÓN SEGURADORA DE CRÉDITO À EXPORTAÇÃO S.A',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('1007',
  'SABEMI SEGURADORA SA',
  '1163',
  'SABEMI',
  'SABEMI | SABEMI SEGURADORA SA',
  'SABEMI',
  'SABEMI',
  'Sabemi',
  'Vida/previdência',
  'Sabemi',
  'Alta',
  None,
  'NÃO'),
 ('37052',
  'BTG Pactual Resseguradora S.A.',
  '1201',
  'BANCO PACTUAL',
  'BANCO PACTUAL | BTG PACTUAL RESSEGURADORA S.A.',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('47741',
  'TOKIO MARINE & NICHIDO FIRE INSURANCE CO. LTD',
  '1222',
  'TOKIO MARINE',
  'TOKIO MARINE | TOKIO MARINE & NICHIDO FIRE INSURANCE CO. LTD',
  'TOKIO MARINE',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'NÃO'),
 ('10138',
  'BAMÉRCIO SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BAMÉRCIO SEGUROS S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5533',
  'ATLÂNTICA COMPANHIA DE SEGUROS',
  '27',
  'BRADESCO',
  'BRADESCO | ATLÂNTICA COMPANHIA DE SEGUROS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('6793',
  'Gente Seguradora S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | GENTE SEGURADORA S.A.',
  'OUTROS',
  'GENTE SEGURADORA',
  'Gente Seguradora',
  'Seguradora nacional independente',
  'Gente Seguradora',
  'Alta',
  None,
  'SIM'),
 ('70939',
  'Nausch Hogan & Murray Brasil Corretora De Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NAUSCH HOGAN & MURRAY BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1848',
  'VINCI VIDA E PREVIDÊNCIA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VINCI VIDA E PREVIDÊNCIA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('42307',
  'XL INSURANCE COMPANY SE',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XL INSURANCE COMPANY SE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5908',
  'GENERALI  BRASIL  SEGUROS S.A.',
  '213',
  'GENERALLI',
  'GENERALLI | GENERALI  BRASIL  SEGUROS S.A.',
  'GENERALI',
  'GENERALI',
  'Generali',
  'Seguradora global multilinha',
  'Generali',
  'Alta',
  None,
  'NÃO'),
 ('78743',
  'Aon Brasil Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AON BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2542',
  'Omint Seguros SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | OMINT SEGUROS SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3794',
  'BS2 SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BS2 SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74667',
  'LOCKTON RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LOCKTON RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3638',
  'SKY SEGURADORA S.A. - Microsseguradora',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SKY SEGURADORA S.A. - MICROSSEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2771',
  'SUDASEG SEGURADORA DE DANOS E PESSOAS S.A. - MICROSSEGURADORA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | SUDASEG SEGURADORA DE DANOS E PESSOAS S.A. - MICROSSEGURADORA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4618',
  'ITAÚ SEGUROS CORPORATIVOS S.A. (Em alteração para ITAÚ XL SEGUROS CORPORATIVO...',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ SEGUROS CORPORATIVOS S.A. (EM ALTERAÇÃO PARA ITAÚ XL SEGUROS CORP...',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('4618',
  'ITAÚ SEGUROS CORPORATIVOS S.A (Em alteração para ITAÚ XL SEGUROS CORPORATIVOS...',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ SEGUROS CORPORATIVOS S.A (EM ALTERAÇÃO PARA ITAÚ XL SEGUROS CORPO...',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('4618',
  'ITAÚ XL SEGUROS CORPORATIVOS S.A.',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ XL SEGUROS CORPORATIVOS S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('1864',
  'REAG SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | REAG SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('23540',
  'CAPEMISA APLUB CAPITALIZAÇÃO S. A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CAPEMISA APLUB CAPITALIZAÇÃO S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4936',
  'COMPANHIA CAPITAL DE SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMPANHIA CAPITAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6301',
  'COMPANHIA BRASILEIRA DE SEGUROS E PREVIDÊNCIA (Em aprovação "RIO GRANDE SEGUR...',
  '825',
  'ICATU',
  'ICATU | COMPANHIA BRASILEIRA DE SEGUROS E PREVIDÊNCIA (EM APROVAÇÃO "RIO GRAN...',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('1015',
  'ALLIANZ BRASIL SEGURADORA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ALLIANZ BRASIL SEGURADORA S.A',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('2763',
  'CESCE Brasil Seguros de Crédito S.A. "Em Aprovação" (Nova Denominação da Segu...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CESCE BRASIL SEGUROS DE CRÉDITO S.A. "EM APROVAÇÃO" (NOVA DENO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6751',
  'SEGUROS SURA S.A."em aprovação"  (ANTIGA ROYAL & SUNALLIANCE SEGUROS (BRASIL)...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEGUROS SURA S.A."EM APROVAÇÃO"  (ANTIGA ROYAL & SUNALLIANCE S...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4634',
  'PATER SEGUROS S.A. "EM APROVAÇÃO" (ANTIGA JAVA NORDESTE SEGUROS S.A.)',
  '1227',
  'SINAF',
  'SINAF | PATER SEGUROS S.A. "EM APROVAÇÃO" (ANTIGA JAVA NORDESTE SEGUROS S.A.)',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SINAF',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5118',
  'TRADITIO COMPANHIA DE SEGUROS \x96 \x93em aprovação\x94 (antiga Sul América Companhia ...',
  None,
  None,
  '| TRADITIO COMPANHIA DE SEGUROS \x96 \x93EM APROVAÇÃO\x94 (ANTIGA SUL AMÉRICA COMPANH...',
  'OUTROS',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'SIM'),
 ('1805',
  'OXXY SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | OXXY SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5151',
  'TOKIO MARINE BRASIL SEGURADORA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | TOKIO MARINE BRASIL SEGURADORA S/A',
  'TOKIO MARINE',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'NÃO'),
 ('5215',
  'ITAÚ VIDA E PREVIDÊNCIA S.A. (antiga ITA',
  '35',
  'ITAÚ',
  'ITAÚ | ITAÚ VIDA E PREVIDÊNCIA S.A. (ANTIGA ITA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5258',
  'Habitasul Seguradora S/A',
  '914',
  'HABITASUL',
  'HABITASUL | HABITASUL SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'HABITASUL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5291',
  'S¦o Paulo Cia Nacional de Seguros Gerais',
  '124',
  'SÃO PAULO',
  'SÃO PAULO | S¦O PAULO CIA NACIONAL DE SEGUROS GERAIS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SÃO PAULO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5312',
  'BRADESCO AUTO/RE COMPANHIA DE SEGUROS "EM APROVAÇÃO" ("ANTIGA" UNIÃO NOVO HAM...',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO AUTO/RE COMPANHIA DE SEGUROS "EM APROVAÇÃO" ("ANTIGA" UNI...',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5355',
  'AXA SEGUROS BRASIL S/A.',
  '299',
  'UAP',
  'UAP | AXA SEGUROS BRASIL S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('5401',
  'PQ SEGUROS "em aprovação "(antiga BBM CO',
  '272',
  'SEGUROS DA BAHIA',
  'SEGUROS DA BAHIA | PQ SEGUROS "EM APROVAÇÃO "(ANTIGA BBM CO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SEGUROS DA BAHIA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5878',
  'CIA SULINA DE PREVIDENCIA E SEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CIA SULINA DE PREVIDENCIA E SEGUROS',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5967',
  'AVS SEGURADORA S/A',
  '1074',
  'SAMCIL',
  'SAMCIL | AVS SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SAMCIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5991',
  'Seguradora Brasileira Rural S/A.',
  '1229',
  'UBF',
  'UBF | SEGURADORA BRASILEIRA RURAL S/A.',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6017',
  'CENTAURO SEGURADORA S/A',
  '1091',
  'CENTAURO SEGURADORA',
  'CENTAURO SEGURADORA | CENTAURO SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CENTAURO SEGURADORA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6050',
  'MAXLIFE SEG DO BR S/A (EM APROV) ANTIGA',
  '1023',
  'NOBRE',
  'NOBRE | MAXLIFE SEG DO BR S/A (EM APROV) ANTIGA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'NOBRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6106',
  'HSBC BAMERINDUS SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | HSBC BAMERINDUS SEGUROS S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6122',
  'INA SEGURADORA S.A.',
  '116',
  'CIGNA',
  'CIGNA | INA SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CIGNA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6157',
  'MINAS BRASIL SEG.VIDA E PREVIDÊNCIA S.A',
  '1207',
  'BANCO MERCANTIL DO BRASIL',
  'BANCO MERCANTIL DO BRASIL | MINAS BRASIL SEG.VIDA E PREVIDÊNCIA S.A',
  'OUTROS',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'SIM'),
 ('6173',
  'INVESTPREV SEGUROS E PREVIDÊNCIA S/A',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | INVESTPREV SEGUROS E PREVIDÊNCIA S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6173',
  'INVESTPREV SEGUROS E PREVID-NCIA S/A',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | INVESTPREV SEGUROS E PREVID-NCIA S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6190',
  'TOKIO MARINE SEGURADORA S.A',
  '1222',
  'TOKIO MARINE',
  'TOKIO MARINE | TOKIO MARINE SEGURADORA S.A',
  'TOKIO MARINE',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'CANADA LIFE PACTUAL PREV. E SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CANADA LIFE PACTUAL PREV. E SEGUROS S/A',
  'BTG PACTUAL',
  'BTG PACTUAL',
  'BTG Pactual',
  'Bancassurance / banco privado',
  'BTG/Pactual',
  'Alta',
  None,
  'NÃO'),
 ('6271',
  'SUDAMERIS VIDA E PREVIDENCIA S/A"em aprovação"9SUDAMERIS GENERALI CIA.NAC.SEG...',
  '86',
  'REAL',
  'REAL | SUDAMERIS VIDA E PREVIDENCIA S/A"EM APROVAÇÃO"9SUDAMERIS GENERALI CIA....',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVIDÊNCIA S.A.',
  '78',
  'BRASIL',
  'BRASIL | AGF VIDA E PREVIDÊNCIA S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('6432',
  'NATIONALE NEDERLANDEN',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NATIONALE NEDERLANDEN',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6483',
  'Winterthur International Brasil Segurado',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | WINTERTHUR INTERNATIONAL BRASIL SEGURADO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6483',
  'Wintrthur International Brasil Segurador',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | WINTRTHUR INTERNATIONAL BRASIL SEGURADOR',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6602',
  'Mitsui Marine & Kyoei Fire Seguros S/A',
  '442',
  'CONCÓRDIA',
  'CONCÓRDIA | MITSUI MARINE & KYOEI FIRE SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CONCÓRDIA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6840',
  'Unibanco AIG SA Seguros e Previdencia',
  '701',
  'PREVER',
  'PREVER | UNIBANCO AIG SA SEGUROS E PREVIDENCIA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('9938',
  'Safra Seguros S.A.',
  '345',
  'SAFRA',
  'SAFRA | SAFRA SEGUROS S.A.',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDENCIA  "em aprovação" (antiga SOCIEDADE AUXILIADORA)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDENCIA  "EM APROVAÇÃO" (ANTIGA SOCIEDADE AUXI...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('12467',
  'Matone Previdência Privada S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MATONE PREVIDÊNCIA PRIVADA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('20451',
  'GLOBAL CAPITALIZACAO S/A',
  '1226',
  'INTERBRAZIL',
  'INTERBRAZIL | GLOBAL CAPITALIZACAO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INTERBRAZIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21946',
  'Sul América Capitalização S.A.',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA CAPITALIZAÇÃO S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('27138',
  'MOTRIN CAPITALIZAÇÃO S/A.',
  '825',
  'ICATU',
  'ICATU | MOTRIN CAPITALIZAÇÃO S/A.',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('28134',
  'HSBC FINANCIAL CAPITALIZAÇÃO (BRASIL) S.',
  '1204',
  'HSBC',
  'HSBC | HSBC FINANCIAL CAPITALIZAÇÃO (BRASIL) S.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('37729',
  'MUNCHENER RÜCK DO BRASIL RESSEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MUNCHENER RÜCK DO BRASIL RESSEGURADORA S.A.',
  'OUTROS',
  'MUNICH RE',
  'Munich Re',
  'Resseguro',
  'Munich Re',
  'Alta',
  None,
  'SIM'),
 ('48879',
  'Kölnische Rückversicherungs-Gesellschaft AG',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | KÖLNISCHE RÜCKVERSICHERUNGS-GESELLSCHAFT AG',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('70203',
  'CIRCLES GROUP BRASIL CORRETORA DE RESSEGUROS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CIRCLES GROUP BRASIL CORRETORA DE RESSEGUROS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('74667',
  'Colemont Brasil Corretagem de Resseguro Ltda.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COLEMONT BRASIL CORRETAGEM DE RESSEGURO LTDA.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6912',
  'ITAUSEG SEGURADORA S.A., "em aprovação" (antiga ITAÚ BMG SEGURADORA S.A.)',
  '35',
  'ITAÚ',
  'ITAÚ | ITAUSEG SEGURADORA S.A., "EM APROVAÇÃO" (ANTIGA ITAÚ BMG SEGURADORA S.A.)',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('39764',
  'MARKEL RESSEGURADORA DO BRASIL S.A. (NOVA DENOMINIAÇÃO DA ALTERRA RESSEGURADO...',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MARKEL RESSEGURADORA DO BRASIL S.A. (NOVA DENOMINIAÇÃO DA ALT...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4669',
  'FAIRFAX BRASIL SEGUROS CORPORATIVOS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | FAIRFAX BRASIL SEGUROS CORPORATIVOS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5070',
  'Santander Seguros S/A (em aprov.Bozano)',
  '1200',
  'SANTANDER',
  'SANTANDER | SANTANDER SEGUROS S/A (EM APROV.BOZANO)',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('6211',
  'Aliança do Brasil Seguros S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALIANÇA DO BRASIL SEGUROS S.A',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('10847',
  'PREVIMIL PREVIDÊNCIA COMPLEMENTAR S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVIMIL PREVIDÊNCIA COMPLEMENTAR S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5444',
  'BRADESCO SEGUROS S.A',
  '27',
  'BRADESCO',
  'BRADESCO | BRADESCO SEGUROS S.A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('37729',
  'MUNICH RE DO BRASIL RESSEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MUNICH RE DO BRASIL RESSEGURADORA S.A.',
  'OUTROS',
  'MUNICH RE',
  'Munich Re',
  'Resseguro',
  'Munich Re',
  'Alta',
  None,
  'SIM'),
 ('1414',
  'Berkley International do Brasil Seguros S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BERKLEY INTERNATIONAL DO BRASIL SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('46507',
  'AXIS Re SE',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | AXIS RE SE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2933',
  'CARDIF DO BRASIL SEGUROS E GARANTIAS S/A',
  '1203',
  'CARDIF',
  'CARDIF | CARDIF DO BRASIL SEGUROS E GARANTIAS S/A',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('3328',
  'Coface do Brasil Seguros de Crédito Interno S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COFACE DO BRASIL SEGUROS DE CRÉDITO INTERNO S/A',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('5096',
  'PHENIX SEGURADORA S.A.',
  '400',
  'PHENIX DE PORTO ALEGRE',
  'PHENIX DE PORTO ALEGRE | PHENIX SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'PHENIX DE PORTO ALEGRE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('72460',
  'GLOBAL RISK BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | GLOBAL RISK BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5282',
  'PRUDENTIAL DO BRASIL SEGUROS DE VIDA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PRUDENTIAL DO BRASIL SEGUROS DE VIDA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('40851',
  'ACE TEMPEST REINSURANCE LTD.',
  '1189',
  'ACE',
  'ACE | ACE TEMPEST REINSURANCE LTD.',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('5193',
  'CIA SEGUROS PREVIDENCIA DO SUL',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CIA SEGUROS PREVIDENCIA DO SUL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2674',
  'ZENPLA SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ZENPLA SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('33294',
  'MAPFRE RE DO BRASIL CIA DE RESSEGUROS',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE RE DO BRASIL CIA DE RESSEGUROS',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('2119',
  'ARUANA SEGURADORA S. A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ARUANA SEGURADORA S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('45896',
  'GARD MARINE & ENERGY LIMITED',
  '27',
  'BRADESCO',
  'BRADESCO | GARD MARINE & ENERGY LIMITED',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5665',
  'MAPFRE VIDA S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VIDA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('4782',
  'NEWE SEGUROS S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NEWE SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('25429',
  'RIO GRANDE CAPITALIZAÇÃO S.A.',
  '825',
  'ICATU',
  'ICATU | RIO GRANDE CAPITALIZAÇÃO S.A.',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('40126',
  'XL RE EUROPE SE',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | XL RE EUROPE SE',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('42684',
  'ARGO RE LTD. ESCRITÓRIO DE REPRESENTAÇÃO NO BRASIL LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARGO RE LTD. ESCRITÓRIO DE REPRESENTAÇÃO NO BRASIL LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76767',
  'ALPER RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALPER RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4111',
  'JNS SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | JNS SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3328',
  'Coface do Brasil Seguros de Crédito S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | COFACE DO BRASIL SEGUROS DE CRÉDITO S/A',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('3786',
  'ARCA SEGURADORA S.A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ARCA SEGURADORA S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4782',
  'MARKEL SEGURADORA DO BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MARKEL SEGURADORA DO BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6033',
  'PORTO SEGURO VIDA E PREVIDÊNCIA S/A.',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO VIDA E PREVIDÊNCIA S/A.',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('2542',
  'OMINT SEGUROS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | OMINT SEGUROS S.A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2020',
  'ATRADIUS CRÉDITO Y CAUCIÓN SEGURADORA S.A',
  '1231',
  'CRÉDITO Y CAUCIÓN',
  'CRÉDITO Y CAUCIÓN | ATRADIUS CRÉDITO Y CAUCIÓN SEGURADORA S.A',
  'OUTROS',
  'ATRADIUS',
  'Atradius Crédito y Caución',
  'Crédito e garantia',
  'Atradius / Crédito y Caución',
  'Alta',
  None,
  'SIM'),
 ('2143',
  'ASSURANT SEGURADORA SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ASSURANT SEGURADORA SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6513',
  'CHUBB SEGUROS BRASIL S.A. (em aprovação)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CHUBB SEGUROS BRASIL S.A. (EM APROVAÇÃO)',
  'CHUBB',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'BB Seguridade',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('10014',
  'ACVAT- PREVIDENCIA PRIVADA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | ACVAT- PREVIDENCIA PRIVADA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4359',
  'Euler do Brasil Seguros S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | EULER DO BRASIL SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2763',
  'Cesce Brasil Seguros de Crédito S.A. "Em Aprovação" (Nova Denominação da Segu...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CESCE BRASIL SEGUROS DE CRÉDITO S.A. "EM APROVAÇÃO" (NOVA DENO...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('49000',
  'GENERAL INSURANCE CORPORATION OF INDIA ESCRITORIO DE REPRESENTACAO  NO BRASIL...',
  '78',
  'BRASIL',
  'BRASIL | GENERAL INSURANCE CORPORATION OF INDIA ESCRITORIO DE REPRESENTACAO  ...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BRASIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('8737',
  'CHARTIS SEGUROS BRASIL S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CHARTIS SEGUROS BRASIL S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5126',
  'UNIVERSAL COMPANHIA DE SEGUROS GERAIS',
  '94',
  'FINASA',
  'FINASA | UNIVERSAL COMPANHIA DE SEGUROS GERAIS',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5355',
  'AXA SEGUROS BRASIL S/A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AXA SEGUROS BRASIL S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('5355',
  'AZUL COMPANHIA DE SEGUROS GERAIS "Em Aprovação" (Antiga AXA SEGUROS BRASIL S/A.)',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | AZUL COMPANHIA DE SEGUROS GERAIS "EM APROVAÇÃO" (ANTIGA AXA SE...',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('5401',
  'PQ SEGUROS (BBM COMPANHIA DE SEGUROS) -',
  '272',
  'SEGUROS DA BAHIA',
  'SEGUROS DA BAHIA | PQ SEGUROS (BBM COMPANHIA DE SEGUROS) -',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SEGUROS DA BAHIA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5665',
  'MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ VIDA E PREVIDÊNCIA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('5843',
  'Indiana Seguros S/A',
  '27',
  'BRADESCO',
  'BRADESCO | INDIANA SEGUROS S/A',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('5878',
  'SULINA SEGURADORA S.A.',
  '1221',
  'SULINA',
  'SULINA | SULINA SEGURADORA S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5941',
  'RELIANCE NATIONAL BRASIL SEGUROS S/A.',
  '1058',
  'BMC',
  'BMC | RELIANCE NATIONAL BRASIL SEGUROS S/A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'BMC',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5967',
  'AVS SEGURADORA S/A (EM APROVAÇÃO-SAMCIL)',
  '1074',
  'SAMCIL',
  'SAMCIL | AVS SEGURADORA S/A (EM APROVAÇÃO-SAMCIL)',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SAMCIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6157',
  'MINAS BRASIL VE+CULOS SEGURADORA S/A',
  '141',
  'MINAS-BRASIL',
  'MINAS-BRASIL | MINAS BRASIL VE+CULOS SEGURADORA S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('6220',
  'Sul América Seguros de Vida e Previdência S.A.',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SEGUROS DE VIDA E PREVIDÊNCIA S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6238',
  'MAPFRE VERA CRUZ SEGURADORA S/A (EM APROVAÇÃO - ANTIGA VERA CRUZ SEGURADORA S...',
  '191',
  'VERA CRUZ',
  'VERA CRUZ | MAPFRE VERA CRUZ SEGURADORA S/A (EM APROVAÇÃO - ANTIGA VERA CRUZ ...',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6297',
  'AIG LIFE COMPANHIA DE SEGUROS',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | AIG LIFE COMPANHIA DE SEGUROS',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6483',
  'XL Insurance (Brazil) Seguradora S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | XL INSURANCE (BRAZIL) SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6696',
  'SUL AMÉRICA COMPANHIA DE SEGUROS GERAIS "em aprovação" (antiga GERLING SUL AM...',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA COMPANHIA DE SEGUROS GERAIS "EM APROVAÇÃO" (ANTIGA ...',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6769',
  'UBF GARANTIAS & SEGUROS S/A',
  '1229',
  'UBF',
  'UBF | UBF GARANTIAS & SEGUROS S/A',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6769',
  'UBF GARANTIAS & SEGUROS S/A',
  '1171',
  'UNITED',
  'UNITED | UBF GARANTIAS & SEGUROS S/A',
  'AMIL',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6777',
  'APS SEGURADORA S.A. (em aprovação) CAOA',
  '1202',
  'CAOA',
  'CAOA | APS SEGURADORA S.A. (EM APROVAÇÃO) CAOA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CAOA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6858',
  'SEGURADORA ROMA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SEGURADORA ROMA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6858',
  'MARES - MAPFRE RISCOS ESPECIAIS SEGURADORA S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MARES - MAPFRE RISCOS ESPECIAIS SEGURADORA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6912',
  'CONAPP CIA NACIONAL DE SEGUROS',
  '850',
  'CONAPP',
  'CONAPP | CONAPP CIA NACIONAL DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CONAPP',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('8141',
  'CAIXAPREV VIDA E PREVID-NCIA S.A.',
  '230',
  'CAIXA ECONÔMICA S/A',
  'CAIXA ECONÔMICA S/A | CAIXAPREV VIDA E PREVID-NCIA S.A.',
  'CAIXA SEGURIDADE',
  'CAIXA SEGURIDADE / CAIXA',
  'Caixa Seguridade e parcerias',
  'Bancassurance / banco público',
  'Entidades e grupo Caixa Econômica',
  'Alta',
  None,
  'NÃO'),
 ('9938',
  'Safra Vida e Prêvidencia S.A. (Safra Seguros S.A.) em Aprovação',
  '345',
  'SAFRA',
  'SAFRA | SAFRA VIDA E PRÊVIDENCIA S.A. (SAFRA SEGUROS S.A.) EM APROVAÇÃO',
  'SAFRA',
  'SAFRA',
  'Safra',
  'Bancassurance / banco privado',
  'Safra',
  'Alta',
  None,
  'NÃO'),
 ('10081',
  'NEWPREV PREVIDÊNCIA PRIVADA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | NEWPREV PREVIDÊNCIA PRIVADA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10138',
  'BAMÉRCIO S/A PREVIDÊNCIA PRIVADA',
  '1196',
  'BAMÉRCIO',
  'BAMÉRCIO | BAMÉRCIO S/A PREVIDÊNCIA PRIVADA',
  'BRADESCO SEGUROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'NÃO'),
 ('10847',
  'PREVIMIL SOC DE PREVIDÊNCIA PRIVADA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PREVIMIL SOC DE PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10979',
  'AUXILIADORA PREVIDÊNCIA "EM APROVAÇÃO" ( antiga SOCIEDADE AUXILIADORA))',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AUXILIADORA PREVIDÊNCIA "EM APROVAÇÃO" ( ANTIGA SOCIEDADE AUXI...',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11045',
  'BRASILPREV PREVIDÊNCIA PRIVADA S/A.',
  '78',
  'BRASIL',
  'BRASIL | BRASILPREV PREVIDÊNCIA PRIVADA S/A.',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('11096',
  'Upofa - União Previdencial',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UPOFA - UNIÃO PREVIDENCIAL',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('19291',
  'Nossa Caixa Seguros e Previdência S.A. "Em Aprovação" (Antiga Nossa Caixa Pre...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | NOSSA CAIXA SEGUROS E PREVIDÊNCIA S.A. "EM APROVAÇÃO" (ANTIGA ...',
  'CAIXA SEGURIDADE',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Nossa Caixa legado',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Média',
  'Banco Nossa Caixa é legado histórico; não tratar como Caixa Econômica. Validar período societário.',
  'SIM'),
 ('22268',
  'AMÉRICA CAPITALIZAÇÃO S.A.',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | AMÉRICA CAPITALIZAÇÃO S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('41432',
  'Odyssey America Reinsurance Corporation Escr Rep no Brasil LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ODYSSEY AMERICA REINSURANCE CORPORATION ESCR REP NO BRASIL LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78875',
  'CAPITAL RE CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CAPITAL RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6106',
  'KIRTON SEGUROS S.A. (Atual denominação da HSBC Seguros (Brasil) S.A.)',
  '27',
  'BRADESCO',
  'BRADESCO | KIRTON SEGUROS S.A. (ATUAL DENOMINAÇÃO DA HSBC SEGUROS (BRASIL) S.A.)',
  'BRADESCO SEGUROS',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'SIM'),
 ('4561',
  'BRASILPREV NOSSO FUTURO SEGUROS E PREVIDÊNCIA S/A',
  '78',
  'BRASIL',
  'BRASIL | BRASILPREV NOSSO FUTURO SEGUROS E PREVIDÊNCIA S/A',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilprev',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('3964',
  'Liberty Mutual Surety Brasil Seguros S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LIBERTY MUTUAL SURETY BRASIL SEGUROS S/A',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('1007',
  'SABEMI SEGURADORA SA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SABEMI SEGURADORA SA',
  'SABEMI',
  'SABEMI',
  'Sabemi',
  'Vida/previdência',
  'Sabemi',
  'Alta',
  None,
  'NÃO'),
 ('5011',
  'Chubb do Brasil Cia de Seguros',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CHUBB DO BRASIL CIA DE SEGUROS',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('5819',
  'AMERICAN LIFE COMPANHIA DE SEGUROS',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AMERICAN LIFE COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6921',
  'Investprev Seguradora S.A',
  '710',
  'RURAL',
  'RURAL | INVESTPREV SEGURADORA S.A',
  'OUTROS',
  'KOVR',
  'Kovr',
  'Seguradora nacional independente',
  'Kovr/Investprev',
  'Alta',
  None,
  'SIM'),
 ('10065',
  'APLUB - Previdência Privada',
  '990',
  'APLUB',
  'APLUB | APLUB - PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'APLUB',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('10898',
  'RECÍPROCA ASSISTÊNCIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | RECÍPROCA ASSISTÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11070',
  'União Previdenciária Cometa do Brasil - COMPREV',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIÃO PREVIDENCIÁRIA COMETA DO BRASIL - COMPREV',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('15580',
  'BMC PREVIDÊNCIA PRIVADA S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BMC PREVIDÊNCIA PRIVADA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('42871',
  'LIBERTY MUTUAL INSURANCE COMPANY',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LIBERTY MUTUAL INSURANCE COMPANY',
  'LIBERTY',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('72796',
  'ESPECIALIZADA RE CORRETORA DE RESSEGUROS',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ESPECIALIZADA RE CORRETORA DE RESSEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3298',
  'Mapfre Previdência S.A.',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE PREVIDÊNCIA S.A.',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6360',
  'Kyoei do Brasil Companhia de Seguros',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | KYOEI DO BRASIL COMPANHIA DE SEGUROS',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('32875',
  'ZURICH RESSEGURADORA BRASIL S/A',
  '531',
  'ZURICH',
  'ZURICH | ZURICH RESSEGURADORA BRASIL S/A',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('5401',
  'PQ SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PQ SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1741',
  'CAPEMISA SEGURADORA DE RAMOS ELEMENTARES S/A',
  '1188',
  'CAPEMI',
  'CAPEMI | CAPEMISA SEGURADORA DE RAMOS ELEMENTARES S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CAPEMI',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('2801',
  'UNIÃO SEGURADORA S/A - VIDA E PREVIDÊNCIA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UNIÃO SEGURADORA S/A - VIDA E PREVIDÊNCIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1571',
  'HDI-GERLING SEGUROS INDUSTRIAIS S.A.',
  '1230',
  'TALANX AG',
  'TALANX AG | HDI-GERLING SEGUROS INDUSTRIAIS S.A.',
  'HDI SEGUROS',
  'HDI / TALANX',
  'HDI / Liberty-Yelum / Indiana / Hannover',
  'Seguradora global multilinha',
  'HDI/Talanx + Liberty/Indiana/Yelum',
  'Alta',
  None,
  'NÃO'),
 ('42463',
  'LLOYD´S',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LLOYD´S',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5096',
  'PHENIX SEGURADORA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PHENIX SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78875',
  'CAPITAL RE CORRETORA DE RESSEGUROS E SERVIÇOS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CAPITAL RE CORRETORA DE RESSEGUROS E SERVIÇOS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4812',
  'Euler Hermes Seguros de Credito à Exportação S.A."em aprovação" (antiga) Eule...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | EULER HERMES SEGUROS DE CREDITO À EXPORTAÇÃO S.A."EM APROVAÇÃO...',
  'EULER HERMES',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('20281',
  'HORIZONTE CAPITALIZAÇÃO S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | HORIZONTE CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('45004',
  'ROYAL & SUN ALLIANCE INSURANCE PLC',
  '515',
  'SUN ALLIANCE',
  'SUN ALLIANCE | ROYAL & SUN ALLIANCE INSURANCE PLC',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SUN ALLIANCE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5118',
  'SUL AMÉRICA CIA NACIONAL DE SEGUROS',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA CIA NACIONAL DE SEGUROS',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6017',
  'CENTAURO VIDA E PREVIDÊNCIA S. A.',
  '1091',
  'CENTAURO SEGURADORA',
  'CENTAURO SEGURADORA | CENTAURO VIDA E PREVIDÊNCIA S. A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'CENTAURO SEGURADORA',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('47848',
  'Berkley Insurance Company',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BERKLEY INSURANCE COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('77747',
  'CAPSICUM RE LATIN AMERICA CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CAPSICUM RE LATIN AMERICA CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('1881',
  'ARCESP SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARCESP SEGURADORA S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3123',
  'VERBIN SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | VERBIN SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('25585',
  'CNP CAPITALIZAÇÃO S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CNP CAPITALIZAÇÃO S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('73415',
  'ARG BRASIL CORRETORA DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARG BRASIL CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('28878',
  'PORTO SEGURO CAPITALIZAÇÃO S/A',
  '51',
  'PORTO SEGURO',
  'PORTO SEGURO | PORTO SEGURO CAPITALIZAÇÃO S/A',
  'PORTO SEGURO',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'NÃO'),
 ('79863',
  'LATIN AMERICA RE ADMINISTRACAO E CORRETAGEM DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | LATIN AMERICA RE ADMINISTRACAO E CORRETAGEM DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4367',
  'PRUDENTIAL DO BRASIL VIDA EM GRUPO S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PRUDENTIAL DO BRASIL VIDA EM GRUPO S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('4715',
  'CREFISA SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | CREFISA SEGUROS S.A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('79766',
  'MDS-RE  Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MDS-RE  CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('76767',
  'Alper Re Corretora de Resseguros Ltda',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALPER RE CORRETORA DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3816',
  'Real Vida e Previdência S.A.',
  '86',
  'REAL',
  'REAL | REAL VIDA E PREVIDÊNCIA S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('4715',
  'BAMERCIO SEGUROS S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BAMERCIO SEGUROS S.A.',
  'OUTROS',
  'BRADESCO SEGUROS',
  'Bradesco e legados',
  'Bancassurance / banco privado',
  'Bradesco e marcas/bancos incorporados',
  'Alta',
  None,
  'SIM'),
 ('2551',
  'CONECTA SEGURADORA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | CONECTA SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('3387',
  'Angelus Seguros SA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ANGELUS SEGUROS SA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6106',
  'HSBC SEGUROS (BRASIL)  S.A.',
  '1204',
  'HSBC',
  'HSBC | HSBC SEGUROS (BRASIL)  S.A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('1775',
  'ACE SEGUROS SOLUÇÕES CORPORATIVAS S/A',
  '1189',
  'ACE',
  'ACE | ACE SEGUROS SOLUÇÕES CORPORATIVAS S/A',
  'CHUBB',
  'CHUBB',
  'Chubb / ACE / Federal',
  'Seguradora global multilinha',
  'Chubb e ACE',
  'Alta',
  None,
  'NÃO'),
 ('6785',
  'Companhia de Seguros Aliança do Brasil',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMPANHIA DE SEGUROS ALIANÇA DO BRASIL',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6238',
  'MAPFRE VERA CRUZ SEGURADORA S/A',
  '1212',
  'MAPFRE',
  'MAPFRE | MAPFRE VERA CRUZ SEGURADORA S/A',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6238',
  'MAPFRE SEGUROS GERAIS S.A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MAPFRE SEGUROS GERAIS S.A',
  'MAPFRE',
  'MAPFRE',
  'Mapfre / Vera Cruz / Mapfre RE',
  'Seguradora global multilinha',
  'Mapfre e Vera Cruz, excluindo entidades BB explicitamente identificadas',
  'Alta',
  None,
  'NÃO'),
 ('6653',
  'PANAMERICANA DE SEGUROS S-A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PANAMERICANA DE SEGUROS S-A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('44814',
  'Partner Reinsurance Europe Public Limited  Company',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PARTNER REINSURANCE EUROPE PUBLIC LIMITED  COMPANY',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5151',
  'TOKIO MARINE BRASIL SEGURADORA S/A',
  '311',
  'AMÉRICA LATINA',
  'AMÉRICA LATINA | TOKIO MARINE BRASIL SEGURADORA S/A',
  'TOKIO MARINE',
  'TOKIO MARINE',
  'Tokio Marine',
  'Seguradora global multilinha',
  'Tokio Marine',
  'Alta',
  None,
  'NÃO'),
 ('5177',
  'AGF BRASIL SEGUROS S.A.',
  '78',
  'BRASIL',
  'BRASIL | AGF BRASIL SEGUROS S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('5231',
  'BCS SEGUROS S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | BCS SEGUROS S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5231',
  'BANCRED SEGURADORA S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | BANCRED SEGURADORA S.A.',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('5355',
  'AZUL COMPANHIA DE SEGUROS GERAIS "Em Aprovação" (Antiga AXA SEGUROS BRASIL S/A.)',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | AZUL COMPANHIA DE SEGUROS GERAIS "EM APROVAÇÃO" (ANTIGA AXA SE...',
  'OUTROS',
  'PORTO SEGURO',
  'Porto / Azul / Itaú Auto e Residência',
  'Seguradora nacional multilinha',
  'Grupo Porto + Azul + Itaú Auto/Residência',
  'Alta',
  None,
  'SIM'),
 ('5401',
  'PQ SEGUROS "em aprovação "(antiga BBM CO',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | PQ SEGUROS "EM APROVAÇÃO "(ANTIGA BBM CO',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5487',
  'SANTANDER NOROESTE SEGURADORA S/A.',
  '370',
  'NOROESTE',
  'NOROESTE | SANTANDER NOROESTE SEGURADORA S/A.',
  'SANTANDER',
  'SANTANDER / ZURICH SANTANDER',
  'Santander Seguros / Zurich Santander / legados Real-HSBC',
  'Bancassurance / banco privado',
  'Santander, Real, HSBC, BBV e legados',
  'Alta',
  None,
  'NÃO'),
 ('5495',
  'Zurich Minas Brasil Seguros S/A "em aprovação" (antiga Cia. Seguros Minas-Bra...',
  '531',
  'ZURICH',
  'ZURICH | ZURICH MINAS BRASIL SEGUROS S/A "EM APROVAÇÃO" (ANTIGA CIA. SEGUROS ...',
  'ZURICH',
  'ZURICH',
  'Zurich / Minas-Brasil',
  'Seguradora global multilinha',
  'Zurich e legados Minas-Brasil',
  'Alta',
  None,
  'NÃO'),
 ('5746',
  'TREVO BANORTE SEGURADORA S/A',
  '426',
  'TREVO',
  'TREVO | TREVO BANORTE SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'TREVO',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('5878',
  'SULINA SEGURADORA S.A.',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SULINA SEGURADORA S.A.',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('5967',
  'SAMCIL SEGURADORA S/A',
  '1074',
  'SAMCIL',
  'SAMCIL | SAMCIL SEGURADORA S/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SAMCIL',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6211',
  'Santa Catarina Seguros e Previdência S.A Aliança do Brasil Seguros S.A (em ho...',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | SANTA CATARINA SEGUROS E PREVIDÊNCIA S.A ALIANÇA DO BRASIL SEG...',
  'OUTROS',
  'BB SEGURIDADE / BANCO DO BRASIL',
  'Brasilseg / Aliança do Brasil',
  'Bancassurance / banco público',
  'Entidades BB/Brasilprev/Brasilseg/Brasilcap/Nossa Caixa legado',
  'Alta',
  None,
  'SIM'),
 ('6220',
  'Sul América Seguros de Pessoas e Previdência S.A. "em aprovação" (antiga Sul ...',
  '19',
  'SUL AMERICA',
  'SUL AMERICA | SUL AMÉRICA SEGUROS DE PESSOAS E PREVIDÊNCIA S.A. "EM APROVAÇÃO...',
  'SULAMÉRICA',
  "REDE D'OR / SULAMÉRICA",
  'SulAmérica',
  'Saúde/odontologia e seguridade',
  'SulAmérica/Sulina',
  'Alta',
  None,
  'NÃO'),
 ('6301',
  'IH CIA DE SEGUROS E PREVIDÊNCIA "em aprovação" (Antiga denominação da Canadá ...',
  '825',
  'ICATU',
  'ICATU | IH CIA DE SEGUROS E PREVIDÊNCIA "EM APROVAÇÃO" (ANTIGA DENOMINAÇÃO DA...',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('6378',
  'MARÍTIMA VIDA E PREVIDÊNCIA S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | MARÍTIMA VIDA E PREVIDÊNCIA S/A',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6378',
  'NATIONWIDE MARÍTIMA VIDA E PREVIDÊNCIA',
  '1213',
  'NATIONWIDE',
  'NATIONWIDE | NATIONWIDE MARÍTIMA VIDA E PREVIDÊNCIA',
  'SOMPO SEGUROS',
  'SOMPO HOLDINGS',
  'Sompo / Yasuda / Marítima',
  'Seguradora global multilinha',
  'Sompo/Yasuda/Marítima',
  'Média',
  'Para visão atual por ramo, revisar varejo Sompo Consumer adquirido pela HDI; corporativo segue Sompo.',
  'NÃO'),
 ('6394',
  'ITAUPREV VIDA E PREVIDÊNCIA S.A. (antiga AGF VIDA E PREVIDÊNCIA  S.A.)',
  '35',
  'ITAÚ',
  'ITAÚ | ITAUPREV VIDA E PREVIDÊNCIA S.A. (ANTIGA AGF VIDA E PREVIDÊNCIA  S.A.)',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6394',
  'AGF VIDA E PREVID-NCIA PRIVADA S.A.',
  '78',
  'BRASIL',
  'BRASIL | AGF VIDA E PREVID-NCIA PRIVADA S.A.',
  'ALLIANZ',
  'ALLIANZ',
  'Allianz / AGF / Allianz Trade',
  'Seguradora global multilinha',
  'Allianz, AGF e Euler Hermes',
  'Alta',
  None,
  'NÃO'),
 ('6530',
  'COMBINED SEGUROS DO BRASIL S/A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | COMBINED SEGUROS DO BRASIL S/A.',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6548',
  'CARDIF DO BRASIL SEGUROS E PREVIDENCIA S',
  '1203',
  'CARDIF',
  'CARDIF | CARDIF DO BRASIL SEGUROS E PREVIDENCIA S',
  'BNP PARIBAS CARDIF',
  'BNP PARIBAS CARDIF',
  'Cardif / Luizaseg',
  'Bancassurance / affinity',
  'Cardif e Luizaseg',
  'Alta',
  None,
  'NÃO'),
 ('6653',
  'PANAMERICANA DE SEGUROS S-A',
  '1211',
  'GRUPO SILVIO SANTOS',
  'GRUPO SILVIO SANTOS | PANAMERICANA DE SEGUROS S-A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'GRUPO SILVIO SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6726',
  'RIO BRANCO SEGURADORA S/A.',
  '299',
  'UAP',
  'UAP | RIO BRANCO SEGURADORA S/A.',
  'OUTROS',
  'AXA / AXA XL',
  'AXA / AXA XL / UAP',
  'Seguradora global multilinha',
  'AXA/AXA XL e legados UAP',
  'Alta',
  None,
  'SIM'),
 ('6769',
  'UBF GARANTIAS & SEGUROS S/A',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | UBF GARANTIAS & SEGUROS S/A',
  'OUTROS',
  'SWISS RE',
  'Swiss Re Corporate Solutions / UBF',
  'Resseguro / grandes riscos',
  'Swiss Re/UBF',
  'Alta',
  None,
  'SIM'),
 ('6840',
  'Unibanco AIG Previdência SA',
  '1180',
  'UNIBANCO SEGUROS',
  'UNIBANCO SEGUROS | UNIBANCO AIG PREVIDÊNCIA SA',
  'ITAÚ UNIBANCO',
  'ITAÚ UNIBANCO',
  'Itaú / Unibanco / legados',
  'Bancassurance / banco privado',
  'Itaú, Unibanco e legados',
  'Alta',
  None,
  'NÃO'),
 ('6891',
  'SAOEX S.A. SEGURADORA E PREVID-NCIA PRIV',
  '728',
  'SAOEX',
  'SAOEX | SAOEX S.A. SEGURADORA E PREVID-NCIA PRIV',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'SAOEX',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('6971',
  'GOLDEN CROSS SEGURADORA  S/A',
  '876',
  'GOLDEN CROSS',
  'GOLDEN CROSS | GOLDEN CROSS SEGURADORA  S/A',
  'OUTROS',
  'HAPVIDA NOTREDAME',
  'NotreDame Intermédica / Golden Cross',
  'Saúde/operadora',
  'NotreDame/Golden Cross',
  'Média',
  'Validar se entidade regulada SUSEP é seguradora legada ou operação de saúde/odonto.',
  'SIM'),
 ('6971',
  'GOLDEN CROSS SEGURADORA  S/A',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | GOLDEN CROSS SEGURADORA  S/A',
  'OUTROS',
  'HAPVIDA NOTREDAME',
  'NotreDame Intermédica / Golden Cross',
  'Saúde/operadora',
  'NotreDame/Golden Cross',
  'Média',
  'Validar se entidade regulada SUSEP é seguradora legada ou operação de saúde/odonto.',
  'SIM'),
 ('10677',
  'ARC PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ARC PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11088',
  'UNIPREV UNIAO PREVIDENCIARIA',
  '1225',
  'INDEPENDENTE',
  'INDEPENDENTE | UNIPREV UNIAO PREVIDENCIARIA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'INDEPENDENTE',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('11126',
  'PECÚLIO UNIÃO PREVIDÊNCIA PRIVADA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | PECÚLIO UNIÃO PREVIDÊNCIA PRIVADA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('21334',
  'ICATU HARTFORD CAPITALIZAÇÃO S/A',
  '825',
  'ICATU',
  'ICATU | ICATU HARTFORD CAPITALIZAÇÃO S/A',
  'ICATU',
  'ICATU',
  'Icatu / Vanguarda',
  'Vida, previdência e capitalização',
  'Icatu e legados',
  'Alta',
  None,
  'NÃO'),
 ('22110',
  'ALFA CAPITALIZAÇÃO S.A.',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ALFA CAPITALIZAÇÃO S.A.',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('22110',
  'ALFA CAPITALIZAÇÃO S.A.',
  '1192',
  'ALFA',
  'ALFA | ALFA CAPITALIZAÇÃO S.A.',
  'OUTROS',
  'ALFA',
  'Alfa Seguros',
  'Banco/seguradora médio porte',
  'Alfa',
  'Alta',
  None,
  'SIM'),
 ('26026',
  'LIDERANÇA CAPITALIZAÇÃOS/A',
  '1211',
  'GRUPO SILVIO SANTOS',
  'GRUPO SILVIO SANTOS | LIDERANÇA CAPITALIZAÇÃOS/A',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'GRUPO SILVIO SANTOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO'),
 ('78778',
  'ORYPABA RIO ADM E CORRETAGEM DE RESSEGUROS LTDA',
  '99999',
  'OUTROS GRUPOS',
  'OUTROS GRUPOS | ORYPABA RIO ADM E CORRETAGEM DE RESSEGUROS LTDA',
  'OUTROS',
  'OUTROS / INDEPENDENTES',
  'OUTROS GRUPOS',
  'Pulverizado / revisar se material',
  'Sem regra segura na v1',
  'Baixa',
  'Manter como outros até medir materialidade de prêmio/sinistro por entidade.',
  'NÃO')]


In [0]:
# ============================================================
# 3. CREATE DATAFRAME WITH TARGET STRUCTURE
# ============================================================

base_schema = StructType([
    StructField("coenti", StringType(), False),
    StructField("noenti", StringType(), False),
    StructField("cogrupo", StringType(), True),
    StructField("nogrupo", StringType(), True),
    StructField("search_string", StringType(), True),
    StructField("nome_empresa_consolidada_query_original", StringType(), True),
    StructField("grupo_concorrencia_n1", StringType(), True),
    StructField("grupo_concorrencia_n2", StringType(), True),
    StructField("tipo_grupo", StringType(), True),
    StructField("regra_aplicada", StringType(), True),
    StructField("confianca_mapeamento", StringType(), True),
    StructField("observacao", StringType(), True),
    StructField("alterou_vs_query_atual", StringType(), True),
])

df_base = spark.createDataFrame(mapping_data, schema=base_schema)

for column_name in df_base.columns:
    df_base = df_base.withColumn(column_name, F.trim(F.col(column_name).cast("string")))

df_ref = (
    df_base
    .withColumn(
        "grupo_concorrencia_n1",
        F.when(
            F.col("grupo_concorrencia_n1").isNull() | (F.col("grupo_concorrencia_n1") == ""),
            F.lit("OUTROS / INDEPENDENTES")
        ).otherwise(F.col("grupo_concorrencia_n1"))
    )
    .withColumn(
        "grupo_concorrencia_n2",
        F.when(
            F.col("grupo_concorrencia_n2").isNull() | (F.col("grupo_concorrencia_n2") == ""),
            F.coalesce(F.col("nogrupo"), F.lit("OUTROS GRUPOS"))
        ).otherwise(F.col("grupo_concorrencia_n2"))
    )
    .withColumn(
        "tipo_grupo",
        F.when(
            F.col("tipo_grupo").isNull() | (F.col("tipo_grupo") == ""),
            F.lit("OUTROS")
        ).otherwise(F.col("tipo_grupo"))
    )
    .withColumn(
        "confianca_mapeamento",
        F.when(
            F.col("confianca_mapeamento").isNull() | (F.col("confianca_mapeamento") == ""),
            F.lit("NAO_CLASSIFICADO")
        ).otherwise(F.col("confianca_mapeamento"))
    )
    .withColumn(
        "regra_aplicada",
        F.when(
            F.col("regra_aplicada").isNull() | (F.col("regra_aplicada") == ""),
            F.lit("Sem regra segura na v1")
        ).otherwise(F.col("regra_aplicada"))
    )
    .withColumn("vigencia_inicio", F.to_date(F.lit(vigencia_inicio_default)))
    .withColumn("vigencia_fim", F.lit(vigencia_fim_default).cast(DateType()))
    .withColumn("flag_ativo", F.lit(flag_ativo_default).cast(BooleanType()))
    .withColumn("versao_mapeamento", F.lit(versao_mapeamento_default))
    .withColumn("fonte_regra", F.lit(fonte_regra_default))
    .withColumn("dt_processamento", F.current_timestamp().cast(TimestampType()))
    .withColumn(
        "hash_registro",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("coenti"), F.lit("")),
                F.coalesce(F.col("noenti"), F.lit("")),
                F.coalesce(F.col("cogrupo"), F.lit("")),
                F.coalesce(F.col("nogrupo"), F.lit("")),
                F.coalesce(F.col("grupo_concorrencia_n1"), F.lit("")),
                F.coalesce(F.col("grupo_concorrencia_n2"), F.lit("")),
                F.coalesce(F.col("tipo_grupo"), F.lit("")),
                F.coalesce(F.col("regra_aplicada"), F.lit("")),
                F.coalesce(F.col("confianca_mapeamento"), F.lit("")),
                F.coalesce(F.col("observacao"), F.lit("")),
                F.coalesce(F.col("versao_mapeamento"), F.lit("")),
            ),
            256
        )
    )
)

display(df_ref.limit(20))


In [0]:
# ============================================================
# 4. VALIDATIONS
# ============================================================

expected_rows = len(mapping_data)
actual_rows = df_ref.count()

if actual_rows != expected_rows:
    raise Exception(f"Row count mismatch. Expected {expected_rows}, got {actual_rows}.")

required_columns = [
    "coenti",
    "noenti",
    "grupo_concorrencia_n1",
    "grupo_concorrencia_n2",
    "tipo_grupo",
    "confianca_mapeamento",
    "vigencia_inicio",
    "flag_ativo",
    "versao_mapeamento",
    "fonte_regra",
    "dt_processamento",
    "hash_registro",
]

for column_name in required_columns:
    null_count = df_ref.filter(F.col(column_name).isNull()).count()
    if null_count > 0:
        raise Exception(f"Column {column_name} has {null_count} null value(s).")

duplicate_mapping = (
    df_ref
    .groupBy("coenti", "noenti", "cogrupo", "nogrupo")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_mapping.count() > 0:
    display(duplicate_mapping)
    raise Exception("Duplicated mapping found for coenti, noenti, cogrupo and nogrupo.")

print("Validations completed successfully.")


In [0]:
# ============================================================
# 5. WRITE DELTA TABLE
# ============================================================

spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")

(
    df_ref.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Table written successfully: {full_table_name}")
print(f"Write mode: {write_mode}")


In [0]:
# ============================================================
# 6. ADD TABLE AND COLUMN COMMENTS
# ============================================================

column_comments = {
    "coenti": "Código da entidade SUSEP",
    "noenti": "Nome da entidade SUSEP",
    "cogrupo": "Código do grupo econômico SUSEP",
    "nogrupo": "Nome do grupo econômico SUSEP",
    "search_string": "Texto normalizado usado como apoio nas regras de classificação",
    "nome_empresa_consolidada_query_original": "Classificação obtida pela primeira query de CASE WHEN criada no projeto",
    "grupo_concorrencia_n1": "Grupo concorrencial consolidado em maior nível para análise executiva",
    "grupo_concorrencia_n2": "Marca, subgrupo ou nível de drill-down do grupo concorrencial",
    "tipo_grupo": "Classificação qualitativa do grupo, como banco, seguradora, resseguradora, saúde ou independente",
    "regra_aplicada": "Regra ou racional utilizado para classificar a entidade",
    "confianca_mapeamento": "Nível de confiança da classificação proposta",
    "observacao": "Observações complementares sobre a classificação",
    "alterou_vs_query_atual": "Indica se a proposta alterou a classificação da query inicial",
    "vigencia_inicio": "Data inicial de validade da regra",
    "vigencia_fim": "Data final de validade, se houver",
    "flag_ativo": "Indica se o de/para está ativo",
    "versao_mapeamento": "Versão da tabela de de/para",
    "fonte_regra": "Fonte ou racional macro usado para criação da regra",
    "dt_processamento": "Data e hora de carga no lake",
    "hash_registro": "Controle de alteração do registro",
}

def sql_string(value):
    return value.replace("'", "''")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    SET TBLPROPERTIES (
        'comment' = 'Dimensão curada para consolidar entidades e grupos SUSEP em grupos concorrenciais de maior nível.'
    )
""")

for column_name, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {full_table_name} ALTER COLUMN {column_name} COMMENT '{sql_string(comment)}'")

print("Table and column comments added successfully.")


In [0]:
# ============================================================
# 7. POST LOAD CHECK
# ============================================================

df_loaded = spark.table(full_table_name)

print(f"Loaded rows: {df_loaded.count()}")
print(f"Loaded SUSEP entities: {df_loaded.select('coenti').distinct().count()}")
print(f"Loaded SUSEP groups: {df_loaded.select('cogrupo').distinct().count()}")
print(f"Loaded competitive groups: {df_loaded.select('grupo_concorrencia_n1').distinct().count()}")

display(
    df_loaded
    .groupBy("grupo_concorrencia_n1", "tipo_grupo", "confianca_mapeamento")
    .agg(
        F.countDistinct("coenti").alias("qtd_entidades_susep"),
        F.countDistinct("cogrupo").alias("qtd_grupos_susep"),
        F.count("*").alias("qtd_linhas_mapeamento"),
    )
    .orderBy(F.desc("qtd_linhas_mapeamento"), "grupo_concorrencia_n1")
)
